# Lightweight-Fed-NIDS on VeReMi NextGen — 20 clients, sparsity 0.7

Bouayad, Alami, Janati Idrissi & Berrada, *Lightweight Federated Learning for Efficient
Network Intrusion Detection* (IEEE Access 2024) — §III-B: a **zero-shot structured
pruning mask** computed once by the server at initialization (DepGraph grouping, L1 group
importance, Eq. (7)–(8)), then **FedAvg with the unweighted mean 1/N** (Eq. (9),
Algorithms 5–6) over the pruned model. The classifier is **DAGSNet** (395,024 params
unpruned → 35,891 at sparsity 0.7, as computed and printed below); the paper's
ResNet/VGG feature extractor on packet-byte images is deliberately absent: the 66 tabular
features enter DAGSNet directly.

| from the paper | from `knowledge/` and the owner's decisions (2026-09-22) |
|---|---|
| §III-B-1-b: mask from the initialized weights, no data; Torch-Pruning / DepGraph; L1 group norm; "sparsity" = per-layer channel ratio | sparsity **0.7** only; uniform per-layer ratio; final Linear (16 logits) not pruned; pruned channels physically removed; the plan (kept indices) is in `CFG` and in every weights file |
| Alg. 5: θ_j ← θ; E local epochs of gradient descent, no rate given | 1 local epoch, batch 512; AdamW, wd 0.0001, re-created per client per round; lr 0.001 → 1e-05 cosine per round; clip 1.0; fp16 AMP; seed 42 |
| Alg. 6 / Eq. (9): θ^{t+1} = (1/N) Σ_j θ_j^t, all N clients every round | exactly that — **unweighted**, although the α = 0.5 partition has client sizes 870 k .. 5.9 M rows |
| T = 5 rounds in the paper | 2 rounds |

**Evaluation:** every round, the aggregated model on the full 10,761,343-row test set
(rows split between the two GPUs, partial confusion matrices summed) — all 10 metrics —
plus the mean / std / min / max over the 20 clients of each client's training loss and
gradient norm. Every client holds the same model after the broadcast, so the global
model's score IS every client's score.

**Deviations, all deliberate** (full list in `docs/REBUILD.md`): no packet-image backbone;
one sparsity level; AdamW with a per-round cosine schedule where Algorithm 5 writes plain
gradient descent with no rate; 1/N where a size-weighted mean would fit this partition
better; DAGSNet is launch-bound at this size, so the paper's ×2–3 training speed-up is
not expected — the probe measures TA / IA on the T4 and the run records train / eval
seconds per round.

Per-round output is **weights only** (θ^t as `state_dict` tensors with the cfg that
rebuilds the pruned architecture, no torch-pruning needed); `proj/ckpt.py::load_weights`
rebuilds it at any round, and the last cell re-derives every published metric from the
confusion matrices on disk.


In [ ]:
import os, subprocess, sys, time, torch
T0 = time.monotonic()     # session clock: the 12 h cap charges for spawn and compile too.
n = torch.cuda.device_count()
assert n == 2, f"expected 2 GPUs, got {n}. machine_shape must be NvidiaTeslaT4."
for i in range(n):
    cap = torch.cuda.get_device_capability(i)
    assert cap == (7, 5), f"GPU {i} is {cap}, expected (7,5) Tesla T4"
    print(i, torch.cuda.get_device_name(i), cap,
          f"{torch.cuda.get_device_properties(i).total_memory/2**30:.1f} GiB")
print("torch", torch.__version__, "| python", sys.version.split()[0])
# NCCL is unused (FL clients never form a process group) but P2P probing can still hang.
os.environ["NCCL_P2P_DISABLE"] = "1"; os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["TORCHINDUCTOR_COMPILE_THREADS"] = "1"
# torch-pruning computes the server's mask once (proj/prune.py). --no-deps: the image's
# torch is never touched. The rebuild path (proj/model.py) does not need it.
r = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
                    "torch-pruning==1.6.1"], capture_output=True, text=True)
assert r.returncode == 0, r.stdout + r.stderr
import torch_pruning
print("torch-pruning", torch_pruning.__version__, "| torch still", torch.__version__)
os.makedirs("/kaggle/working/proj", exist_ok=True)
open("/kaggle/working/proj/__init__.py", "w").close()
sys.path.insert(0, "/kaggle/working")


In [ ]:
CFG = dict(
    # --- architecture: knowledge/ARCHITECTURE.md, frozen (395,024 params before pruning)
    patch_len=6, stem_ch=96, dense_growth=32, dense_layers=3,
    incep_modules=2, fire_modules=3, dropout=0.1,
    num_classes=16, n_features=66,
    # --- Lightweight-Fed-NIDS: the mask (proj/prune.py) and the local step. The plan,
    #     its id and the pruned parameter count are filled in below by compute_plan.
    sparsity=0.7, prune_importance="group_l1",
    lr=0.001, lr_schedule="cosine", lr_min=1e-05,
    weight_decay=0.0001, local_epochs=1,
    rounds=2, clip=1.0, seed=42,
    # --- compute: knowledge/DATASET.md §4
    n_clients=20, batch=512, eval_batch=16384,
    device="cuda", world_size=2, compile=True,
    # Each client starts a fresh GradScaler at 2**16 and spends a few steps calibrating.
    # Fixed before the first measurement so it cannot be widened afterwards.
    max_skips_per_client=16,
    preds_rounds=[],   # (10.76 M,) uint8 per listed round
    finalize_reserve_seconds=900,   # never start a round that leaves no time to commit it
    run_name="lwfednids_20c_probe",
    cache="/kaggle/temp/veremi_cache",
    max_seconds=2.0 * 3600,   # 12 h hard cap; leave room to finalize artifacts
    require_resume=False,
)
# data_id is filled in below, once the feature order and scaler are known. It is part of
# proj/ckpt.py FINGERPRINT_KEYS, so a run cannot resume across a changed preprocessing.
for k, v in CFG.items(): print(f"{k:>20} = {v}")


In [ ]:
%%writefile /kaggle/working/proj/model.py
"""DAGSNet — Khan et al. 2025 §4.10, Eq. (38)-(48). 395,024 parameters unpruned.
Vào: (B, 66) đặc trưng đã z-score.  Ra: (B, 16) logit (CHƯA softmax).

Lightweight-Fed-NIDS (Bouayad et al. 2024) uses ONE global model for every client. The
paper's ResNet/VGG feature extractor on packet-byte images is deliberately absent (owner's
decision, 2026-09-22): the 66 tabular features enter DAGSNet directly, exactly as in
knowledge/ARCHITECTURE.md. `build_model(cfg)` builds that DAGSNet and, when cfg carries a
pruning plan (proj/prune.py), slices it to the pruned architecture the server computed
once at initialization -- so a checkpoint's weights rebuild from cfg alone, without
torch-pruning installed.
"""
import torch
import torch.nn as nn


def cbr(i, o, k):
    """Conv → BatchNorm → ReLU. bias=False vì BatchNorm ngay sau đã có tham số dịch."""
    return nn.Sequential(nn.Conv1d(i, o, k, padding=k // 2, bias=False),
                         nn.BatchNorm1d(o), nn.ReLU(inplace=True))


class DenseNet1d(nn.Module):
    """Eq. (38)-(39): mỗi lớp nhận nối của toàn bộ feature map trước đó."""
    def __init__(self, cin, growth, layers):
        super().__init__()
        self.blocks = nn.ModuleList([cbr(cin + i * growth, growth, 3) for i in range(layers)])
        self.out_ch = cin + layers * growth

    def forward(self, x):
        for b in self.blocks:
            x = torch.cat([x, b(x)], dim=1)                     # Eq. (38)
        return x                                                # Eq. (39)


class Inception1d(nn.Module):
    """Eq. (40): bốn nhánh song song 1x1 / 3x3 / 5x5 / pool, nối lại."""
    def __init__(self, cin, c):
        super().__init__()
        self.b1 = cbr(cin, c, 1)
        self.b3 = nn.Sequential(cbr(cin, c, 1), cbr(c, c, 3))
        self.b5 = nn.Sequential(cbr(cin, c, 1), cbr(c, c, 5))
        self.bp = nn.Sequential(nn.MaxPool1d(3, 1, 1), cbr(cin, c, 1))
        self.out_ch = 4 * c

    def forward(self, x):
        return torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], dim=1)


class GoogleNet1d(nn.Module):
    """Eq. (40)-(41): các inception module xếp chồng."""
    def __init__(self, cin, modules_n, c=32):
        super().__init__()
        mods, ch = [], cin
        for _ in range(modules_n):
            m = Inception1d(ch, c); mods.append(m); ch = m.out_ch
        self.net = nn.Sequential(*mods); self.out_ch = ch

    def forward(self, x):
        return self.net(x)


class AlexNet1d(nn.Module):
    """Eq. (42)-(44). ceil_mode=True: trục vị trí chỉ dài 11, không được để pool co về 0."""
    def __init__(self, cin, ch=128):
        super().__init__()
        self.net = nn.Sequential(
            cbr(cin, ch, 3), nn.MaxPool1d(2, ceil_mode=True),
            cbr(ch, ch, 3),  nn.MaxPool1d(2, ceil_mode=True),
            cbr(ch, ch, 3))
        self.out_ch = ch

    def forward(self, x):
        return self.net(x)


class Fire1d(nn.Module):
    """Eq. (45)-(46): squeeze 1x1 nuôi hai nhánh expand 1x1 và 3x3."""
    def __init__(self, cin, sq, ex):
        super().__init__()
        self.squeeze = cbr(cin, sq, 1)                          # Eq. (46)
        self.e1 = cbr(sq, ex, 1)
        self.e3 = cbr(sq, ex, 3)
        self.out_ch = 2 * ex

    def forward(self, x):
        s = self.squeeze(x)
        return torch.cat([self.e1(s), self.e3(s)], dim=1)       # Eq. (45)


class SqueezeNet1d(nn.Module):
    def __init__(self, cin, modules_n, sq=32, ex=48):
        super().__init__()
        mods, ch = [], cin
        for _ in range(modules_n):
            m = Fire1d(ch, sq, ex); mods.append(m); ch = m.out_ch
        self.net = nn.Sequential(*mods); self.out_ch = ch

    def forward(self, x):
        return self.net(x)


class DAGSNet(nn.Module):
    def __init__(self, cfg, n_features, out_dim):
        super().__init__()
        self.patch_len = cfg["patch_len"]
        assert n_features % self.patch_len == 0
        self.k = n_features // self.patch_len                   # 11
        cin = self.patch_len                                    # 6 kênh

        s = cfg["stem_ch"]
        self.stems   = nn.ModuleList([cbr(cin, s, 1) for _ in range(4)])
        self.dense   = DenseNet1d(s, cfg["dense_growth"], cfg["dense_layers"])
        self.google  = GoogleNet1d(s, cfg["incep_modules"])
        self.alex    = AlexNet1d(s)
        self.squeeze = SqueezeNet1d(s, cfg["fire_modules"])
        comb = self.dense.out_ch + self.google.out_ch + self.alex.out_ch + self.squeeze.out_ch

        self.head = nn.Sequential(                              # Eq. (48)
            nn.LayerNorm(comb), nn.Dropout(cfg["dropout"]),
            nn.Linear(comb, 256), nn.ReLU(inplace=True),
            nn.Dropout(cfg["dropout"]), nn.Linear(256, out_dim))

    def forward(self, x):                                       # (B, n_features)
        # view rồi MỚI transpose: gom 6 cột liên tiếp thành một patch, sau đó patch
        # mới trở thành trục vị trí. Làm view(B, 6, 11) thẳng sẽ trộn sai các cột.
        Fm = x.view(x.shape[0], self.k, self.patch_len).transpose(1, 2)   # (B, 6, 11)
        feats = [gp(stem(Fm)) for stem, gp in
                 zip(self.stems, [self.dense, self.google, self.alex, self.squeeze])]
        pooled = [f.mean(dim=-1) for f in feats]                # global average pool
        return self.head(torch.cat(pooled, dim=1))              # Eq. (47) -> (48)


# Cấu hình ĐÚNG như knowledge/ARCHITECTURE.md. Đổi bất kỳ giá trị nào ở đây thì
# state_dict sẽ không nạp được — đó là chủ ý.
CFG = {
    "patch_len": 6, "stem_ch": 96, "dense_growth": 32, "dense_layers": 3,
    "incep_modules": 2, "fire_modules": 3, "dropout": 0.1, "num_classes": 16,
}
N_PARAMS_FULL = 395_024       # the unpruned DAGSNet, head 256 -> 16


def build_full(cfg):
    """The unpruned DAGSNet of knowledge/ARCHITECTURE.md. Consumes the default RNG in the
    same order every time, so `torch.manual_seed(seed); build_full(cfg)` is theta_0."""
    return DAGSNet({k: cfg[k] for k in CFG}, n_features=cfg["n_features"],
                   out_dim=cfg["num_classes"])


def build_model(cfg):
    """The model every client and the server train: the unpruned DAGSNet when cfg has no
    pruning plan, otherwise that DAGSNet sliced by `cfg['prune_plan']` (Eq. (8):
    theta' = M ⊙ theta_0, with the zeroed channels physically removed as the paper does
    with DepGraph). cfg is the dict saved in every checkpoint, so a model rebuilt here
    matches the one that produced those weights -- and needs only torch to do so."""
    m = build_full(cfg)
    plan = cfg.get("prune_plan")
    if plan:
        from proj.prune import apply_plan
        m = apply_plan(m, plan)
    return m


def n_params(model):
    return sum(p.numel() for p in model.parameters())


In [ ]:
%%writefile /kaggle/working/proj/prune.py
"""The pruning mask of Lightweight-Fed-NIDS (Bouayad et al. 2024, §III-B-1-b), computed
ONCE on the server at initialization and never again.

    theta_0  <- seeded initialization of the unpruned DAGSNet (knowledge/ARCHITECTURE.md)
    M        <- zero-shot structured mask: DepGraph groups the layers by their inter- and
                intra-layer dependencies (Fang et al. 2023), the importance of prunable
                dimension k of group g is the L1 group norm (Eq. (7), I(theta) = ||theta||_1),
                and the `sparsity` fraction of the LEAST important channels of every layer
                is removed (uniform per-layer ratio, no data, no gradients)
    theta'   <- M ⊙ theta_0                                                     Eq. (8)
                with the zeroed channels physically removed ("DeepGraph" in the paper),
                so the model is genuinely smaller instead of sparse

The mask is a property of the architecture and the seed, not of any client's data, which
is what lets the server compute it without seeing the data and send it once.

Two representations, one truth:
  * torch-pruning computes the mask and slices a model (`compute_plan`, needs the
    `torch_pruning` package -- only at server initialization);
  * the PLAN it produced -- for every parametrised module, the ORIGINAL indices of the
    channels kept on its output and input dimension -- is recorded as plain lists, and
    `apply_plan` rebuilds the identical pruned module from an unpruned DAGSNet with torch
    alone. `compute_plan` asserts the two agree tensor for tensor before returning, so a
    checkpoint saved with the plan in its cfg rebuilds anywhere.

`sparsity` is torch-pruning's `pruning_ratio`: the fraction of CHANNELS removed per layer
(the paper used Torch-Pruning and calls the same knob "sparsity"). Parameters fall by
roughly (1 - s)^2 for an inner layer since both its input and its output shrink: at
s = 0.7 DAGSNet goes from 395,024 to 35,891 parameters (measured, torch-pruning 1.6.1).
"""
import copy, hashlib, json
import torch
import torch.nn as nn

from proj.model import build_full, n_params

PRUNABLE = (nn.Conv1d, nn.BatchNorm1d, nn.LayerNorm, nn.Linear)
IMPORTANCE = "group_l1"            # Eq. (7): L1 norm per prunable dimension of the group


def _dims(mod):
    """(out_size, in_size) of the two prunable dimensions; in_size None for a module with a
    single feature dimension (BatchNorm, LayerNorm)."""
    if isinstance(mod, nn.Conv1d):
        return mod.out_channels, mod.in_channels
    if isinstance(mod, nn.Linear):
        return mod.out_features, mod.in_features
    if isinstance(mod, nn.BatchNorm1d):
        return mod.num_features, None
    if isinstance(mod, nn.LayerNorm):
        return mod.normalized_shape[0], None
    raise TypeError(type(mod))


def compute_plan(cfg, verbose=False):
    """The server's initialization step. Returns (plan, info): `plan` is None when
    cfg['sparsity'] == 0 (the unpruned baseline), else {module_name: {"out": [...],
    "in": [...] | None}} with the ORIGINAL indices kept; `info` carries the parameter and
    MAC counts before and after, plus the per-layer channel table.

    Deterministic: theta_0 is `torch.manual_seed(cfg['seed']); build_full(cfg)`, exactly
    what proj/driver.py builds as the initial global model, so theta' = M ⊙ theta_0 holds
    channel for channel."""
    s = float(cfg["sparsity"])
    if not 0.0 <= s < 1.0:
        raise ValueError(f"sparsity {s} not in [0, 1)")
    torch.manual_seed(cfg["seed"])
    full = build_full(cfg).eval()
    x = torch.zeros(2, cfg["n_features"])
    info = {"sparsity": s, "importance": IMPORTANCE, "n_params_full": n_params(full)}
    if s == 0.0:
        info.update(n_params=info["n_params_full"], plan_id=None)
        return None, info

    import torch_pruning as tp                     # server-side only
    macs_full, _ = tp.utils.count_ops_and_params(full, x)
    ref = copy.deepcopy(full)                       # theta_0, untouched
    names = {id(m): n for n, m in full.named_modules()}
    # kept[name] = {"out": [original idx still present, in current order], "in": [...]}
    kept = {n: {"out": list(range(_dims(m)[0])),
                "in": list(range(_dims(m)[1])) if _dims(m)[1] is not None else None}
            for n, m in full.named_modules() if isinstance(m, PRUNABLE)}
    imp = tp.importance.GroupMagnitudeImportance(p=1)          # ||.||_1 per channel
    pruner = tp.pruner.BasePruner(full, x, importance=imp, pruning_ratio=s,
                                  ignored_layers=[full.head[5]],  # keep the 16 logits
                                  global_pruning=False)         # uniform ratio per layer
    for group in pruner.step(interactive=True):
        # idxs of every dependency in a group are in the coordinates of the model AS IT IS
        # NOW; collect them, prune, then translate through the current->original maps.
        todo = {}
        for dep, idxs in group:
            mod = dep.target.module
            if not isinstance(mod, PRUNABLE):
                continue                                        # cat / mean / relu / pool
            name = names[id(mod)]
            h = getattr(dep.handler, "__name__", "")
            if "in_channels" in h and kept[name]["in"] is not None:
                dim = "in"
            elif "out_channels" in h or "in_channels" in h:
                dim = "out"                       # BN / LN: one feature dimension
            else:
                raise RuntimeError(f"unknown pruning handler {h!r} on {name}")
            todo.setdefault((name, dim), set()).update(int(i) for i in idxs)
        group.prune()
        for (name, dim), drop in todo.items():
            cur = kept[name][dim]
            if max(drop) >= len(cur):
                raise RuntimeError(f"{name}.{dim}: index {max(drop)} outside {len(cur)}")
            kept[name][dim] = [v for j, v in enumerate(cur) if j not in drop]
    plan = {n: {"out": kept[n]["out"], "in": kept[n]["in"]} for n in kept}
    # ---- the plan must rebuild EXACTLY what torch-pruning built, tensor for tensor
    mine = apply_plan(copy.deepcopy(ref), plan)
    a, b = mine.state_dict(), full.state_dict()
    if list(a) != list(b):
        raise RuntimeError("apply_plan and torch-pruning disagree on the state_dict keys")
    for k in a:
        if a[k].shape != b[k].shape or not torch.equal(a[k], b[k]):
            raise RuntimeError(f"apply_plan and torch-pruning disagree at {k}: "
                               f"{tuple(a[k].shape)} vs {tuple(b[k].shape)}")
    with torch.no_grad():
        if not torch.equal(mine(x), full(x)):
            raise RuntimeError("rebuilt pruned model and torch-pruning's differ in forward")
    macs_pruned, _ = tp.utils.count_ops_and_params(full, x)
    table = {}
    for n, m in ref.named_modules():
        if isinstance(m, PRUNABLE):
            o, i = _dims(m)
            table[n] = {"type": type(m).__name__, "out": [o, len(plan[n]["out"])],
                        "in": [i, len(plan[n]["in"])] if i is not None else None}
    info.update(n_params=n_params(full), macs_full=int(macs_full), macs_pruned=int(macs_pruned),
                plan_id=plan_id(plan), layers=table,
                torch_pruning=getattr(tp, "__version__", "?"))
    if verbose:
        print(f"[prune] sparsity {s}: {info['n_params_full']:,} -> {info['n_params']:,} params "
              f"({info['n_params'] / info['n_params_full']:.3%}), MACs {macs_full:,} -> "
              f"{macs_pruned:,} ({macs_pruned / macs_full:.3%}), plan {info['plan_id']}")
    return plan, info


def plan_id(plan):
    """Sixteen hex chars over the canonical JSON of the plan: what the fingerprint sees."""
    if plan is None:
        return None
    return hashlib.sha256(json.dumps(plan, sort_keys=True).encode()).hexdigest()[:16]


@torch.no_grad()
def apply_plan(model, plan):
    """Slice an UNPRUNED model to the plan, module by module, keeping the listed original
    indices. Every module in the plan must exist with the unpruned dimensions; an index
    outside them or a module the plan does not know is an error, never a silent skip."""
    have = {n for n, m in model.named_modules() if isinstance(m, PRUNABLE)}
    if set(plan) != have:
        raise RuntimeError(f"plan modules {sorted(set(plan) ^ have)} do not match the model")
    for name, sel in plan.items():
        mod = model.get_submodule(name)
        o, i = _dims(mod)
        out = torch.as_tensor(sel["out"], dtype=torch.long)
        inn = torch.as_tensor(sel["in"], dtype=torch.long) if sel["in"] is not None else None
        if out.numel() == 0 or (out.max() >= o) or (inn is not None and (inn.numel() == 0 or inn.max() >= i)):
            raise RuntimeError(f"{name}: plan indices outside the module's dimensions")
        if (inn is None) != (i is None):
            raise RuntimeError(f"{name}: plan has an 'in' list for a single-dimension module")
        dev, dt = mod.weight.device, mod.weight.dtype
        if isinstance(mod, nn.Conv1d):
            new = nn.Conv1d(len(inn), len(out), mod.kernel_size[0], stride=mod.stride[0],
                            padding=mod.padding[0], bias=mod.bias is not None,
                            device=dev, dtype=dt)
            new.weight.copy_(mod.weight[out][:, inn])
            if mod.bias is not None: new.bias.copy_(mod.bias[out])
        elif isinstance(mod, nn.Linear):
            new = nn.Linear(len(inn), len(out), bias=mod.bias is not None, device=dev, dtype=dt)
            new.weight.copy_(mod.weight[out][:, inn])
            if mod.bias is not None: new.bias.copy_(mod.bias[out])
        elif isinstance(mod, nn.BatchNorm1d):
            new = nn.BatchNorm1d(len(out), eps=mod.eps, momentum=mod.momentum,
                                 affine=mod.affine, track_running_stats=mod.track_running_stats,
                                 device=dev, dtype=dt)
            new.weight.copy_(mod.weight[out]); new.bias.copy_(mod.bias[out])
            new.running_mean.copy_(mod.running_mean[out])
            new.running_var.copy_(mod.running_var[out])
            new.num_batches_tracked.copy_(mod.num_batches_tracked)
        elif isinstance(mod, nn.LayerNorm):
            new = nn.LayerNorm(len(out), eps=mod.eps, elementwise_affine=mod.elementwise_affine,
                               device=dev, dtype=dt)
            new.weight.copy_(mod.weight[out]); new.bias.copy_(mod.bias[out])
        parent, _, child = name.rpartition(".")
        setattr(model.get_submodule(parent) if parent else model, child, new)
    return model


In [ ]:
%%writefile /kaggle/working/proj/ckpt.py
"""Weights, resume state and the completion marker — three files, one atomic round.

The per-round weights file holds WEIGHTS ONLY and loads with weights_only=True: the
server's aggregate theta^t as plain state_dict tensors, plus the cfg that rebuilds the
(pruned) architecture -- including the pruning plan, so `build_model(cfg)` needs nothing
but torch. RNG and the round counter live in a separate resume bundle keyed by the same
round, so a reader never has to unpickle arbitrary objects to look at a checkpoint.

Persistent state of Lightweight-Fed-NIDS is: the round and theta^t. The mask is fixed at
initialization (it is the architecture itself), every client starts each round from
theta^t, and AdamW is re-created per client per round (owner's decision, see
lwfednids.py), so no optimizer state exists at a round boundary.
"""
import csv, hashlib, json, os, random, shutil
from pathlib import Path
import numpy as np, torch

SUBDIRS = ("weights", "resume", "complete", "metrics", "preds", "confusion", "reports", "logs")

# What a later session imports. `preds` and `logs` are not needed to CONTINUE, but a run
# split across sessions must still be verifiable from its final output alone.
RESUME_SUBDIRS = ("weights", "resume", "metrics", "confusion", "preds", "logs")

# A run tree may start at a round r0 > 1 when the previous sessions were handed over as a
# HANDOFF BUNDLE: only round r0's artifacts, plus this file, which carries the sha256 of
# every weights file 1..r0 taken from the full tree the owner holds locally. Round r0 is
# then anchored two ways — its own bytes must hash to chain[r0], and its prev_sha must be
# chain[r0-1] — so a bundle cannot be assembled from two runs of the same config, and the
# full chain is re-verified locally after the sessions are merged (merge_sessions.py).
HANDOFF = "reports/handoff.json"

# Every input that changes what the numbers mean. `rounds` IS in the list since the
# per-round LR schedule (proj.lwfednids.lr_at) spans the whole run: the weights at round r
# depend on how many rounds were planned, so a continuation push must plan the same
# total. `sparsity`, `plan_id` and `prune_importance` are here because the mask IS the
# architecture: two runs with different masks are different models, and `plan_id` hashes
# the exact channel indices the server kept. Paths, world_size, compile, eval_batch,
# max_seconds and require_resume are operational and absent.
FINGERPRINT_KEYS = (
    "patch_len", "stem_ch", "dense_growth", "dense_layers", "incep_modules",
    "fire_modules", "dropout", "num_classes", "n_features",
    "sparsity", "plan_id", "prune_importance",
    "lr", "lr_schedule", "lr_min", "rounds", "weight_decay", "clip",
    "n_clients", "batch", "local_epochs", "seed",
    "data_id", "run_name",
)


def run_dir(run_name, base="/kaggle/working/runs"):
    d = Path(base) / run_name
    for s in SUBDIRS: (d / s).mkdir(parents=True, exist_ok=True)
    return d


def fingerprint(cfg):
    """A missing key is a KeyError, never a default. Silently hashing `None` for a key that
    was renamed is exactly how a fingerprint stops protecting anything."""
    missing = [k for k in FINGERPRINT_KEYS if k not in cfg]
    if missing:
        raise KeyError(f"fingerprint needs {missing} in CFG; add them, do not default them")
    payload = json.dumps({k: cfg[k] for k in FINGERPRINT_KEYS}, sort_keys=True, default=str)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


def file_sha(path):
    """Hash of the file as written. Not a re-serialisation: torch.save embeds a zip whose
    bytes are not reproducible, so only the bytes on disk are a stable identity."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def atomic_save(obj, path):
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp); os.replace(tmp, path)     # replace is atomic on POSIX


def atomic_np_save(path, arr):
    """np.save appends .npy to a name that lacks it, so write through a handle."""
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "wb") as f:
        np.save(f, arr); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)


# --- RNG kept as tensors and primitives so the resume bundle also loads weights_only=True.
def rng_state():
    npy = np.random.get_state()
    return {"python": random.getstate(),
            "numpy": (npy[0], torch.from_numpy(npy[1].copy()), int(npy[2]), int(npy[3]),
                      float(npy[4])),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def set_rng_state(s):
    random.setstate(tuple(s["python"]))
    n = s["numpy"]
    np.random.set_state((n[0], n[1].numpy().astype(np.uint32), n[2], n[3], n[4]))
    torch.set_rng_state(s["torch"].cpu())
    if s.get("cuda") is not None: torch.cuda.set_rng_state_all(s["cuda"])


def cpu_sd(model):
    """A detached CPU copy of a module's state_dict: tensors only, so the file it goes
    into loads with weights_only=True."""
    core = getattr(model, "_orig_mod", model)                 # unwrap torch.compile
    return {k: v.detach().cpu().clone() for k, v in core.state_dict().items()}


# --------------------------------------------------------------------------- write
def save_round_weights(global_sd, rnd, cfg, metrics, d):
    """global_sd: state_dict of theta^t; `metrics` its 10 metrics on the full test set.
    The caller writes the marker, and only after every other artifact is on disk."""
    prev = d / "weights" / f"round_{rnd - 1:03d}.pt"
    # The hash of the weights this round was trained FROM. Two runs of the same config have
    # the same fingerprint, so without this an import can keep round 1 of run A and take
    # round 2 of run B and call the result one training history.
    prev_sha = file_sha(prev) if rnd > 1 and prev.is_file() else None
    atomic_save({"round": int(rnd),
                 "global": global_sd,
                 "cfg": {k: v for k, v in cfg.items()},       # rebuild recipe, plan included
                 "fingerprint": fingerprint(cfg),
                 "prev_sha": prev_sha,
                 "metrics": metrics},
                d / "weights" / f"round_{rnd:03d}.pt")
    # The RNG here is the DRIVER's, and the driver does not train. Recorded for forensics:
    # worker training RNG is re-derived from (seed, round, client), so continuation does
    # not depend on restoring this.
    atomic_save({"round": int(rnd), "rng": rng_state(),
                 "note": "no optimizer state: AdamW is re-created per client per round; "
                         "worker RNG derives from (seed, round, client)",
                 "fingerprint": fingerprint(cfg)},
                d / "resume" / f"round_{rnd:03d}.pt")
    return d / "weights" / f"round_{rnd:03d}.pt"


def mark_complete(d, rnd):
    (d / "complete" / f"round_{rnd:03d}.done").write_text("")


# --------------------------------------------------------------------------- rebuild
def _load_into(model, sd, expect_params):
    sd = {k.removeprefix("module.").removeprefix("_orig_mod."): v for k, v in sd.items()}
    bad = [k for k, v in sd.items() if v.is_floating_point() and not torch.isfinite(v).all()]
    if bad:
        raise RuntimeError(f"non-finite values in checkpoint tensors {bad[:3]}")
    model.load_state_dict(sd, strict=True)
    n = sum(p.numel() for p in model.parameters())
    if expect_params is not None and n != expect_params:
        raise RuntimeError(f"rebuilt {n:,} parameters, expected {expect_params:,}")
    return model.eval()


def load_weights(path, build_model, expect_params=None, device="cpu"):
    """Rebuild theta^t at exactly this checkpoint, from weights alone. Returns (G, raw).

    Every assert here exists because its failure is otherwise silent: strict=True catches a
    filtered running_mean/var (eval() would then normalize by 0/1 and report nothing), the
    parameter count catches a cfg -- or a pruning plan -- that drifted from the one that
    trained these weights, and the finite check catches a diverged tensor that argmax
    would turn into a plausible label. `expect_params=None` takes the count the file's
    own cfg declares (`n_params`), which the plan must reproduce."""
    ck = torch.load(path, map_location="cpu", weights_only=True)
    if ck.get("fingerprint") != fingerprint(ck["cfg"]):
        raise RuntimeError("weights file fingerprint disagrees with its own cfg")
    if expect_params is None:
        expect_params = ck["cfg"].get("n_params")
    G = _load_into(build_model(ck["cfg"]), ck["global"], expect_params).to(device)
    return G, ck


# --------------------------------------------------------------------------- verify
def read_handoff(d):
    """The handoff record of a tree that starts past round 1, or None. A malformed file
    reads as absent, and an absent record means rounds before the first marker are simply
    missing — which last_complete_round then treats as a gap, never as a start."""
    p = Path(d) / HANDOFF
    if not p.is_file():
        return None
    try:
        h = json.loads(p.read_text())
        h["round"] = int(h["round"])
        h["chain"] = {int(k): str(v) for k, v in h["chain"].items()}
        return h
    except Exception:
        return None


def handoff_record(d, rnd, fp, extra=None):
    """The sha chain 1..rnd of a FULL tree, for a bundle that will carry round rnd alone.
    (theta^t is the whole algorithm state, so a bundle of one round is a complete resume
    point.)
    Refuses a tree whose chain does not verify all the way from round 1. The caller writes
    the dict to HANDOFF inside the bundle, never inside the full tree."""
    d = Path(d)
    if _first_round(d) != 1:
        raise RuntimeError(f"{d} is itself a partial tree; export a handoff from the merged run")
    last = last_complete_round(d, fp)
    if last is None or last < rnd:
        raise RuntimeError(f"{d}: rounds 1..{rnd} do not all verify (last={last}); "
                           "nothing to hand off")
    chain = {str(r): file_sha(d / "weights" / f"round_{r:03d}.pt") for r in range(1, rnd + 1)}
    return {"round": int(rnd), "fingerprint": fp, "chain": chain, **(extra or {})}


def round_ok(d, rnd, fp=None, chain=True):
    """A round counts only if every artifact of that round is present, READABLE, and links
    to the round before it. The marker alone proves nothing. When the round before it is
    not on disk, the link is checked against the handoff chain instead (see HANDOFF), and
    the round's own bytes must match the chain too."""
    w = d / "weights" / f"round_{rnd:03d}.pt"
    r = d / "resume" / f"round_{rnd:03d}.pt"
    m = d / "metrics" / f"round_{rnd:03d}.json"
    c = d / "confusion" / f"round_{rnd:03d}.npy"
    for path in (w, r, m, c):
        if not path.is_file() or path.stat().st_size == 0:
            return False
    try:
        ck = torch.load(w, map_location="cpu", weights_only=True, mmap=True)
        rs = torch.load(r, map_location="cpu", weights_only=True)   # not just "it exists"
        row = json.loads(m.read_text())
        np.load(c)
    except Exception:
        return False                      # truncated or corrupt reads as a failed round
    if int(ck.get("round", -1)) != rnd or int(row.get("round", -1)) != rnd:
        return False
    if int(rs.get("round", -1)) != rnd:
        return False
    if fp is not None and (ck.get("fingerprint") != fp or rs.get("fingerprint") != fp):
        return False
    if chain:
        # Round r is only meaningful as the product of round r-1. Same config, same
        # fingerprint, different training history -> different bytes -> chain breaks here.
        prev = d / "weights" / f"round_{rnd - 1:03d}.pt"
        if rnd > 1 and not prev.is_file():
            h = read_handoff(d)
            if h is None or h["round"] != rnd or (fp is not None and h["fingerprint"] != fp):
                return False                  # a bare round r > 1 is a gap, not a start
            if h["chain"].get(rnd) != file_sha(w) or ck.get("prev_sha") != h["chain"].get(rnd - 1):
                return False
        else:
            want = file_sha(prev) if rnd > 1 and prev.is_file() else None
            if ck.get("prev_sha") != want:
                return False
    return True


def _markers(d):
    return sorted(int(p.stem.split("_")[1]) for p in (d / "complete").glob("round_*.done"))


def _first_round(d):
    """1 for a full tree; the handoff round for a bundle that starts later; the lowest
    marker otherwise (round_ok then rejects it as a gap unless a handoff anchors it)."""
    h = read_handoff(d)
    m = _markers(d)
    if h is not None and not (d / "weights" / f"round_{h['round'] - 1:03d}.pt").is_file():
        return h["round"]
    return m[0] if m else 1


def last_complete_round(d, fp=None):
    """Largest r such that rounds first..r are ALL complete, where first is 1 or the
    handoff round of a bundle. A gap ends the run."""
    last = 0
    m = _markers(d)
    top = m[-1] if m else 0
    for r in range(_first_round(d), top + 1):
        if not (d / "complete" / f"round_{r:03d}.done").is_file() or not round_ok(d, r, fp):
            break
        last = r
    return last or None


# --------------------------------------------------------------------------- resume
def _attached_source(run_name, attached=Path("/kaggle/input")):
    if not attached.exists():
        return None
    roots = sorted({p.parent for p in attached.rglob("complete/round_*.done")
                    if run_name in p.parts})
    if len(roots) > 1:
        raise RuntimeError(f"Multiple resume trees for {run_name}: {roots}")
    return roots[0].parent if roots else None


def _import_from(src, d, fp):
    """Copy through a staging tree, verify there, then publish one round at a time with its
    marker last. Idempotent: a crash mid-publish leaves that round unmarked, and the next
    attempt re-copies it from the still-mounted source. The source's own markers bound the
    import, and the destination must not already hold a different history."""
    stage = d.parent / f".{d.name}.import"
    shutil.rmtree(stage, ignore_errors=True)
    for sub in RESUME_SUBDIRS + ("complete", "reports"):
        (stage / sub).mkdir(parents=True, exist_ok=True)
    for sub in RESUME_SUBDIRS + ("complete",):
        peer = src / sub
        if not peer.is_dir(): continue
        for f in peer.iterdir():
            if f.is_file():
                # copyfile, not copy2: a read-only mount's mode would carry across and the
                # first rewrite would die with PermissionError.
                shutil.copyfile(f, stage / sub / f.name)
                os.chmod(stage / sub / f.name, 0o644)
    if (src / HANDOFF).is_file():
        shutil.copyfile(src / HANDOFF, stage / HANDOFF)
        os.chmod(stage / HANDOFF, 0o644)

    first = _first_round(stage)
    src_last = first - 1
    while ((stage / "complete" / f"round_{src_last + 1:03d}.done").is_file()
           and round_ok(stage, src_last + 1, fp)):
        src_last += 1
    if src_last < first:
        src_last = 0                                   # nothing verifiable, not even round first

    if src_last and first > 1:
        # The destination must not hold rounds the bundle cannot vouch for.
        if any((d / "weights" / f"round_{r:03d}.pt").is_file() for r in range(1, first)):
            shutil.rmtree(stage, ignore_errors=True)
            raise RuntimeError(f"{d} already holds rounds before {first}; a handoff bundle "
                               "cannot be spliced onto a tree that has its own history")
        shutil.copyfile(stage / HANDOFF, d / HANDOFF)
    for r in range(first, src_last + 1):
        here = d / "weights" / f"round_{r:03d}.pt"
        if here.is_file() and file_sha(here) != file_sha(stage / "weights" / f"round_{r:03d}.pt"):
            shutil.rmtree(stage, ignore_errors=True)
            raise RuntimeError(
                f"round {r} in {d} and in {src} have the same config but different weights: "
                "these are two different training runs, not one interrupted one. Refusing to "
                "splice them. Detach one source, or start a new run_name.")

    published = 0
    for r in range(first, src_last + 1):
        if not round_ok(d, r, fp):
            for sub in RESUME_SUBDIRS:
                for f in (stage / sub).glob(f"round_{r:03d}.*"):
                    shutil.copyfile(f, d / sub / f.name)
                    os.chmod(d / sub / f.name, 0o644)
        if not (d / "complete" / f"round_{r:03d}.done").is_file():
            mark_complete(d, r)                     # marker last, per round
        published = r
    shutil.rmtree(stage, ignore_errors=True)
    n_mark = len(list((src / "complete").glob("round_*.done"))) if (src / "complete").is_dir() else 0
    print(f"[resume] {src}: {n_mark} marker(s), verified to round {src_last}, imported {first}..{published}"
          + (f" (handoff bundle: chain 1..{first} attested, verified locally before export)"
             if first > 1 else "")
          if published else
          f"[resume] {src} held no verifiable committed round; starting from 0")
    return published


def resolve_resume(run_name, cfg=None, attached=Path("/kaggle/input"), d=None):
    """working/ first, then any attached input (previous kernel output or a checkpoint
    dataset). Returns the last round that is complete AND verified, or None."""
    d = run_dir(run_name) if d is None else d
    fp = fingerprint(cfg) if cfg is not None else None
    src = _attached_source(run_name, attached)
    if src is not None:
        _import_from(src, d, fp)
    rebuild_history(d, fp)
    return last_complete_round(d, fp)


# --------------------------------------------------------------------------- history
def _write_csv(p, rows):
    tmp = Path(p).with_suffix(".csv.tmp")
    with open(tmp, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader(); w.writerows(rows)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, p)                      # a crash mid-write cannot truncate the live file


def rebuild_history(d, fp=None):
    """history.csv (one row per round: the global model's 10 metrics plus the means over
    clients of the training statistics) and clients.csv (one row per client per round:
    that client's training statistics) are DERIVED from metrics/round_NNN.json, never
    authoritative. Only the verified contiguous range first..last goes in."""
    last = last_complete_round(d, fp) or 0
    rows, crow = [], []
    for r in range(_first_round(d), last + 1):
        p = d / "metrics" / f"round_{r:03d}.json"
        if not p.is_file(): break
        try: j = json.loads(p.read_text())
        except Exception: break
        rows.append({k: v for k, v in j.items() if k not in ("clients", "per_class")})
        crow += [{"round": r, **c} for c in j["clients"]]
    if rows:
        _write_csv(d / "history.csv", rows)
        _write_csv(d / "clients.csv", crow)
    else:
        for n in ("history.csv", "clients.csv"):
            if (d / n).exists(): (d / n).unlink()  # a stale CSV outlives the rounds it described
    return len(rows)


def append_history(d, row, client_rows):
    """Keyed by round: a redone round replaces its lines instead of duplicating them."""
    p = d / "history.csv"
    rows = {}
    if p.exists():
        with open(p) as f:
            rows = {int(r["round"]): r for r in csv.DictReader(f)}
    rows[int(row["round"])] = {k: str(v) for k, v in row.items()}
    _write_csv(p, [rows[k] for k in sorted(rows)])
    q = d / "clients.csv"
    old = []
    if q.exists():
        with open(q) as f:
            old = [r for r in csv.DictReader(f) if int(r["round"]) != int(row["round"])]
    _write_csv(q, old + [{k: str(v) for k, v in c.items()} for c in client_rows])


In [ ]:
%%writefile /kaggle/working/proj/metrics.py
"""All 10 metrics from a full confusion matrix. No batch averaging, no sampling."""
import numpy as np

METRIC_KEYS = ("accuracy", "precision_macro", "precision_micro", "precision_weighted",
               "recall_macro", "recall_micro", "recall_weighted",
               "f1_macro", "f1_micro", "f1_weighted")


def metrics_from_confusion(cm):
    """cm[i, j] = count of true class i predicted as j. Integer counts in, 10 floats out."""
    cm = np.asarray(cm, dtype=np.float64)
    tp = np.diag(cm)
    support = cm.sum(axis=1)                       # true count per class
    pred = cm.sum(axis=0)                          # predicted count per class
    total = cm.sum()

    # A class never predicted has precision 0/0; sklearn defines it as 0 with zero_division=0.
    prec = np.divide(tp, pred, out=np.zeros_like(tp), where=pred > 0)
    rec = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    denom = prec + rec
    f1 = np.divide(2 * prec * rec, denom, out=np.zeros_like(tp), where=denom > 0)

    acc = tp.sum() / total
    w = support / total                            # weighted = support-weighted mean
    # float(), not np.float64: a numpy scalar anywhere in a checkpoint dict makes
    # torch.load(weights_only=True) refuse the whole file, and the failure only appears
    # when something later tries to read it back.
    return {k: float(v) for k, v in (
        ("accuracy", acc),
        ("precision_macro", prec.mean()), ("precision_micro", acc),
        ("precision_weighted", (prec * w).sum()),
        ("recall_macro", rec.mean()), ("recall_micro", acc),
        ("recall_weighted", (rec * w).sum()),
        ("f1_macro", f1.mean()), ("f1_micro", acc),
        ("f1_weighted", (f1 * w).sum()))}


def per_class_from_confusion(cm, class_names):
    cm = np.asarray(cm, dtype=np.float64)
    tp, support, pred = np.diag(cm), cm.sum(axis=1), cm.sum(axis=0)
    prec = np.divide(tp, pred, out=np.zeros_like(tp), where=pred > 0)
    rec = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    d = prec + rec
    f1 = np.divide(2 * prec * rec, d, out=np.zeros_like(tp), where=d > 0)
    return [{"idx": i, "class": class_names[i], "support": int(support[i]),
             "precision": float(prec[i]), "recall": float(rec[i]), "f1": float(f1[i])}
            for i in range(len(class_names))]


In [ ]:
%%writefile /kaggle/working/proj/data.py
"""Parquet -> resident fp16 tensors. One pass per session, then no input pipeline at all.

Two facts from knowledge/DATASET.md that produce silent corruption if ignored:
  * train/ is ALREADY z-scored; test/ is NOT. Applying scaler.json to train a second time
    destroys it and raises nothing.
  * the integer label is in column `label` (int8, 0..15). Decoding `attack_type` strings
    over 43 M rows costs tens of seconds per pass for the same information.
"""
import json
from pathlib import Path
import numpy as np
import pyarrow.dataset as ds

CACHE_FILES = ("train_X.f16.npy", "train_y.u8.npy", "test_X.f16.npy",
               "test_y.u8.npy", "spans.json")


def find_root(sentinel, bases=("/kaggle/input",)):
    """Kaggle mounts are nested by kind and owner; the prefix is not /kaggle/input/<slug>/.
    Resolve by locating the sentinel instead of hard-coding a depth."""
    depth = len(Path(sentinel).parts)          # strip the whole sentinel, not one level
    hits = []
    for b in bases:
        p = Path(b)
        if p.exists():
            for q in p.rglob(sentinel):
                if q.is_dir():
                    r = q
                    for _ in range(depth): r = r.parent
                    hits.append(r)
    hits = sorted(set(hits))
    if len(hits) != 1:
        raise RuntimeError(f"expected exactly one {sentinel!r} under {bases}, got {hits}")
    return hits[0]


def parquet_files(root):
    """The parquet parts of a directory, in a fixed order, and NOTHING else.

    `ds.dataset(dir, format="parquet")` opens every file it finds. The centralized test
    directory ships a `part-NNNNN.stats.json` sidecar next to each part, so `to_table()`
    died on Kaggle with "Parquet magic bytes not found in footer" after a nine-minute train
    decode. It survived locally only because the smoke fixture reads through `to_batches()`
    and breaks early, never reaching a sidecar -- a lazy reader hides exactly this.

    sorted() is not cosmetic either: the fragment order fixes the row order of the test set,
    and therefore the order of y_true and of every saved prediction vector."""
    files = sorted(str(f) for f in Path(root).rglob("*.parquet"))
    if not files:
        raise RuntimeError(f"no .parquet files under {root}")
    return files


def _labels(col, num_classes=16):
    """astype(np.uint8) on a label of -1 gives 255 and on 300 gives 44 -- both are silent,
    and both survive every downstream assert because the counts still add up."""
    v = col.to_numpy(zero_copy_only=False)
    lo, hi = int(v.min()), int(v.max())
    if lo < 0 or hi >= num_classes:
        raise RuntimeError(f"labels out of range [{lo}, {hi}], expected 0..{num_classes-1}")
    return v.astype(np.uint8)


def load_clients(fl_root, feature_cols, n_clients, dtype=np.float16):
    """Returns X (N,66) fp16, y (N,) uint8, and spans[cid] = (lo, hi) contiguous row range.

    Contiguous spans are what make the training loop a slice + randperm instead of a
    gather over a client-id column."""
    dirs = sorted((fl_root / "train").glob("client_id=*"),
                  key=lambda p: int(p.name.split("=")[1]))
    if len(dirs) != n_clients:
        raise RuntimeError(f"expected {n_clients} client dirs, found {len(dirs)}")
    cols = list(feature_cols) + ["label"]
    xs, ys, spans, off = [], [], {}, 0
    for d in dirs:
        cid = int(d.name.split("=")[1])
        t = ds.dataset(parquet_files(d), format="parquet").to_table(columns=cols)
        n = t.num_rows
        a = np.empty((n, len(feature_cols)), dtype=dtype)
        for j, c in enumerate(feature_cols):
            a[:, j] = t.column(c).to_numpy(zero_copy_only=False).astype(dtype, copy=False)
        xs.append(a)
        ys.append(_labels(t.column("label")))
        spans[cid] = (off, off + n); off += n
        del t
    return np.concatenate(xs), np.concatenate(ys), spans


def load_test(test_root, feature_cols, scaler, dtype=np.float16):
    """test/ is raw: apply scaler.json here, and nowhere else."""
    t = ds.dataset(parquet_files(test_root), format="parquet").to_table(
        columns=list(feature_cols) + ["label"])
    n = t.num_rows
    X = np.empty((n, len(feature_cols)), dtype=dtype)
    for j, c in enumerate(feature_cols):
        v = t.column(c).to_numpy(zero_copy_only=False).astype(np.float64)
        v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)
        s = scaler[c]
        X[:, j] = ((v - s["mean"]) / s["std_used"]).astype(dtype)
    y = _labels(t.column("label"))
    return X, y


def assert_fp16_safe(X, name):
    """knowledge/DATASET.md measured max|x| = 570.44 on both splits, two orders below the
    fp16 ceiling. Assert it rather than inherit the assumption."""
    m = float(np.abs(X).max())
    if not np.isfinite(m) or m >= 65504:
        raise RuntimeError(f"{name}: max|x| = {m} is not fp16-safe")
    return m


def cache_ok(cache, want, n_clients, n_features=66):
    """Is the prepack cache complete, current and self-consistent?

    A matching manifest is a claim, not evidence. Running the notebook's own hit branch with
    a matching manifest and no train_X printed "cache reusable"; the worker then died opening
    a file that had never been written. Headers are read through mmap, so this costs a few
    stat calls and no data."""
    cache = Path(cache)
    mf = cache / "manifest.json"
    if not mf.is_file():
        return False
    try:
        if json.loads(mf.read_text()) != want:
            return False
        for f in CACHE_FILES:
            if not (cache / f).is_file() or (cache / f).stat().st_size == 0:
                return False
        spans = {int(k): tuple(v)
                 for k, v in json.load(open(cache / "spans.json")).items()}
        X = np.load(cache / "train_X.f16.npy", mmap_mode="r")
        Y = np.load(cache / "train_y.u8.npy", mmap_mode="r")
        TX = np.load(cache / "test_X.f16.npy", mmap_mode="r")
        TY = np.load(cache / "test_y.u8.npy", mmap_mode="r")
    except Exception as e:
        print("[cache] unreadable:", e)
        return False
    n = sum(hi - lo for lo, hi in spans.values())
    r = sorted(spans.values())
    return bool(
        len(spans) == n_clients
        and X.dtype == np.float16 and Y.dtype == np.uint8
        and TX.dtype == np.float16 and TY.dtype == np.uint8
        and X.shape == (n, n_features) and Y.shape == (n,)
        and TX.ndim == 2 and TX.shape[1] == n_features and len(TX) == len(TY)
        and r and r[0][0] == 0 and r[-1][1] == n
        and all(a[1] == b[0] for a, b in zip(r, r[1:])))


In [ ]:
%%writefile /kaggle/working/proj/lwfednids.py
"""Lightweight-Fed-NIDS (Bouayad, Alami, Janati Idrissi, Berrada — IEEE Access 2024):
the federated loop of §III-B, Algorithm 5 (client) and Algorithm 6 (server), with the
zero-shot pruning mask of §III-B-1-b computed once by the server (proj/prune.py).

Initialization (server, once):
    theta_0  <- seeded DAGSNet                                       §III-B-1-a
    M        <- zero-shot structured mask, L1 group importance      Eq. (7)
    theta'   <- M ⊙ theta_0, pruned channels physically removed     Eq. (8)
    broadcast (theta', M) to every client                           Alg. 6 line 4

Round t, EVERY client j (N = |C|, full participation as in Alg. 6 lines 6-8):
    theta_j <- theta                                                 Alg. 5 line 4
    (the mask is already baked into the architecture: Alg. 5 line 5 is the identity here,
     since a channel that was removed cannot regrow)
    one local epoch over D_j, per batch b:
        L      = CE(f(theta_j, x_b), y_b)                            Eq. (5)/(6), C = 16
        theta_j <- theta_j - eta_t * step(grad L)                    Alg. 5 line 8
    upload theta_j

Server:
    theta^{t+1} = (1/N) sum_{j=1}^{N} theta_j^t                       Eq. (9), Alg. 6 line 9

Deviations from the paper, all decided with the owner on 2026-09-22 and published with
the numbers (docs/REBUILD.md):
  * no feature-extraction backbone (ResNet-50/101, VGG-19 on 40x300-byte flow images):
    the 66 tabular VeReMi features enter the DAGSNet classifier directly;
  * the aggregate is the paper's UNWEIGHTED mean 1/N (Eq. 9), although this partition is
    non-IID with client sizes 870 k .. 5.9 M rows (the paper's IID shards were equal);
  * the local step is AdamW (weight decay 1e-4, knowledge/ARCHITECTURE.md) at a per-ROUND
    cosine learning rate 1e-3 -> 1e-5 over the 50 rounds, constant within a round and
    re-created per client per round, where Algorithm 5 line 8 writes plain gradient
    descent and the paper names no rate;
  * one sparsity level, 0.7 (the paper reports 0 / 0.5 / 0.7 / 0.9);
  * 50 rounds x 1 local epoch, batch 512 / 512 / 256 for 20 / 50 / 100 clients, seed 42.
"""
import contextlib
import math
import torch
import torch.nn.functional as F


def amp(cfg):
    """fp16 autocast on CUDA; a no-op on CPU so the same code runs in the local
    simulation. Never bf16: the T4 is sm_75 and falls back to a slow emulation path."""
    if cfg.get("device", "cuda") == "cuda":
        return torch.autocast("cuda", dtype=torch.float16)
    return contextlib.nullcontext()


# --------------------------------------------------------------------- flat layout
# One flat float vector + one int vector per model. A DAGSNet state_dict has 192 entries;
# torch.multiprocessing gives each tensor its own shared-memory fd, so 100 clients a round
# would exhaust the process fd limit. Parameters come FIRST so vec[:n_params] is exactly
# the learnable block; BN running stats follow as buffers.
def layout(model):
    pnames = {n for n, _ in model.named_parameters()}
    sd = model.state_dict()
    fkeys = [k for k in sd if k in pnames]
    fkeys += [k for k in sd if k not in pnames and sd[k].is_floating_point()]
    ikeys = [k for k in sd if not sd[k].is_floating_point()]
    n_params = sum(sd[k].numel() for k in fkeys if k in pnames)
    return fkeys, ikeys, n_params


def flatten(model, fkeys, ikeys):
    sd = model.state_dict()
    fv = torch.cat([sd[k].reshape(-1).float() for k in fkeys])
    iv = torch.stack([sd[k].reshape(-1).long().squeeze() for k in ikeys]) if ikeys \
        else torch.zeros(0, dtype=torch.long)
    return fv, iv


def unflatten_into(model, fv, iv, fkeys, ikeys):
    sd = model.state_dict()
    o = 0
    for k in fkeys:
        t = sd[k]; n = t.numel()
        t.copy_(fv[o:o + n].view_as(t)); o += n           # copy_ keeps addresses -> CUDA graph valid
    for j, k in enumerate(ikeys):
        sd[k].copy_(iv[j].view_as(sd[k]))
    return model


# --------------------------------------------------------------------- learning rate
LR_SCHEDULES = ("constant", "cosine")


def lr_at(cfg, rnd):
    """Learning rate of round `rnd` (1-based), held constant within the round. A pure
    function of (cfg, rnd): a resumed session applies exactly the value the original
    session would have.

      constant : cfg['lr'] every round
      cosine   : lr_min + (lr - lr_min)/2 * (1 + cos(pi * (rnd-1) / (rounds-1)))
                 -- cfg['lr'] at round 1, cfg['lr_min'] at round cfg['rounds'].
    """
    sched = cfg.get("lr_schedule", "constant")
    if sched == "constant":
        return float(cfg["lr"])
    if sched == "cosine":
        T = int(cfg["rounds"])
        if not 1 <= rnd <= T:
            raise ValueError(f"round {rnd} outside 1..{T}: the cosine schedule is undefined")
        if T == 1:
            return float(cfg["lr"])
        lo, hi = float(cfg["lr_min"]), float(cfg["lr"])
        return lo + 0.5 * (hi - lo) * (1.0 + math.cos(math.pi * (rnd - 1) / (T - 1)))
    raise ValueError(f"lr_schedule {sched!r} not in {LR_SCHEDULES}")


# --------------------------------------------------------------------- the loss
# Order of the per-step values accumulated on the device (see `client_update`).
ACC_KEYS = ("loss", "gnorm")


def ce_loss(z, y):
    """Eq. (5)/(6) for one batch, from fp32 logits: the categorical cross-entropy of the
    softmax over the 16 classes (the paper's binary form is the C = 2 case)."""
    return F.cross_entropy(z, y)


# --------------------------------------------------------------------- client update
def make_optimizer(Se, lr, cfg, fused):
    """AdamW over the (pruned) model's parameters, weight decay from cfg. fused=True
    collapses the step into one multi-tensor kernel on CUDA."""
    return torch.optim.AdamW(list(Se.parameters()), lr=lr,
                             weight_decay=cfg["weight_decay"], fused=fused)


def client_update(Sc, Se, opt, scaler, X, Y, lo, hi, cfg, gen):
    """Algorithm 5 lines 6-9: `local_epochs` passes over rows [lo, hi) of the resident
    tensors for one client.

    Se is the eager module that owns theta_j; Sc the compiled alias (or the same object
    when compile is off). The tail batch is a different shape and would recompile the
    CUDA graph once per client, so it runs on the eager module: same weights, same math.

    Returns (acc, n_steps): `acc` is a device tensor of len(ACC_KEYS) + 2 sums over
    APPLIED steps (the last two entries are the skipped-step count and the count of
    applied steps whose gradient norm was not finite), read once by the caller.
    Dropout reads torch's default generator, which the worker re-seeds from
    (seed, round, client) before calling this; `gen` drives only the shuffles."""
    B, clip, dev = cfg["batch"], cfg["clip"], X.device
    params = list(Se.parameters())
    Se.train()
    acc = torch.zeros(len(ACC_KEYS) + 2, device=dev)
    zeros = torch.zeros(len(ACC_KEYS), device=dev)
    n = 0
    for _ in range(cfg["local_epochs"]):
        perm = lo + torch.randperm(hi - lo, generator=gen, device=dev)
        for i in range(0, hi - lo, B):
            # One captured graph per step; the explicit iteration boundary keeps CUDA-graph
            # trees from treating the next replay as part of this one.
            if X.is_cuda:
                torch.compiler.cudagraph_mark_step_begin()
            idx = perm[i:i + B]
            full = idx.numel() == B
            xb = X[idx].float()
            yb = Y[idx].long()
            with amp(cfg):
                z = (Sc if full else Se)(xb)
            loss = ce_loss(z.float(), yb)                          # loss in fp32
            scaler.scale(loss).backward()
            scaler.unscale_(opt)                                   # grads now in true units
            gn = torch.nn.utils.clip_grad_norm_(params, clip)
            prev = scaler._scale.clone() if scaler.is_enabled() else None
            scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True)
            # A skipped step overflowed: its grad-norm is inf and its loss may be nan.
            # torch.where, not multiplication -- inf*0 is nan.
            applied = (scaler._scale >= prev) if prev is not None \
                else torch.ones((), dtype=torch.bool, device=dev)
            vals = torch.stack([loss.detach(), gn])
            acc[:len(ACC_KEYS)] += torch.where(applied, vals, zeros)
            acc[len(ACC_KEYS)] += (~applied).float()
            acc[len(ACC_KEYS) + 1] += ((~torch.isfinite(gn)) & applied).float()
            n += 1
    return acc, n


def expected_steps(n_k, cfg):
    return cfg["local_epochs"] * math.ceil(n_k / cfg["batch"])


# --------------------------------------------------------------------- server side
def aggregate(updates, n_clients):
    """Eq. (9) / Algorithm 6 line 9: theta^{t+1} = (1/N) sum_{j=1}^N theta_j^t.

    UNWEIGHTED, as the paper writes it -- a client with 870 k rows counts exactly as much
    as one with 5.9 M (docs/REBUILD.md). `updates` must hold every one of the N clients,
    sorted by client id: float addition order decides the result, and it must not be set
    by a completion race. A missing client is an error, never a smaller divisor: dropping
    one changes the participation the paper fixes at all N."""
    cids = [c for c, _, _ in updates]
    if cids != list(range(n_clients)):
        raise ValueError(f"aggregate() needs every client 0..{n_clients - 1} in order, "
                         f"got {cids[:6]}...")
    acc = None
    for cid, fv, iv in updates:
        acc = fv / n_clients if acc is None else acc.add_(fv, alpha=1.0 / n_clients)
    # num_batches_tracked is an int counter, not an averageable quantity; with the default
    # BatchNorm momentum=0.1 it is unused at inference. Take the max so it stays monotone.
    ints = torch.stack([iv for _, _, iv in updates]).amax(dim=0) if updates[0][2].numel() \
        else updates[0][2]
    return acc, ints


In [ ]:
%%writefile /kaggle/working/proj/evaluate.py
"""Evaluation of the global model on the fixed test set, and the two exact speed-ups.

Lightweight-Fed-NIDS keeps ONE global model, so a round's evaluation is a single pass over
the 10,761,343 test rows -- split between the two GPUs by row range (proj/driver.py), each
half producing a partial confusion matrix that the driver sums.

Two exact speed-ups, both measured on 2xT4 in sibling rebuilds of this owner:
  * BatchNorm folding: in eval mode BN is an affine map with constant coefficients, so it
    folds into the preceding Conv1d exactly (max|dlogit| 4.8e-07 measured), removing 31
    kernel launches per forward. The pruned model keeps the cbr structure, so it folds too.
  * One folded TEMPLATE module per worker, compiled once with CUDA graphs. Weights are
    folded and copied INTO the template with load_state_dict (in-place copy_), so
    parameter addresses never change and the captured graph stays valid every round.
"""
import copy
import torch
import torch.nn as nn


@torch.no_grad()
def fold_bn(model):
    """Return an eval-mode copy with every BatchNorm folded into its preceding Conv1d.

        y = gamma*(conv(x) - mean)/sqrt(var + eps) + beta
          = conv'(x) + b'   with   w' = w*gamma/sqrt(var+eps),  b' = beta - gamma*mean/sqrt(var+eps)

    Exact for model.eval(); meaningless for model.train(). Every BatchNorm in DAGSNet sits
    inside a `cbr` block, i.e. Sequential(Conv1d, BN, ReLU), and the assertion at the end
    is what stops a future architecture change from silently leaving one unfolded."""
    m = copy.deepcopy(model).eval()
    for seq in m.modules():
        if not (isinstance(seq, nn.Sequential) and len(seq) >= 2
                and isinstance(seq[0], nn.Conv1d) and isinstance(seq[1], nn.BatchNorm1d)):
            continue
        conv, bn = seq[0], seq[1]
        inv = torch.rsqrt(bn.running_var + bn.eps)
        w = conv.weight * (bn.weight * inv).view(-1, 1, 1)
        b = bn.bias - bn.weight * bn.running_mean * inv
        if conv.bias is not None:
            b = b + conv.bias * bn.weight * inv
        new = nn.Conv1d(conv.in_channels, conv.out_channels, conv.kernel_size[0],
                        stride=conv.stride[0], padding=conv.padding[0], bias=True,
                        device=w.device, dtype=w.dtype)
        new.weight.copy_(w)
        new.bias.copy_(b)
        seq[0] = new
        seq[1] = nn.Identity()
    left = [n for n, mod in m.named_modules() if isinstance(mod, nn.BatchNorm1d)]
    assert not left, f"BatchNorm survived folding at {left}; the cbr pattern changed"
    for p in m.parameters():
        p.requires_grad_(False)
    return m.eval()


def load_folded(template, model):
    """Fold `model` and copy the result into `template` in place (same folded structure).
    strict=True: a key mismatch means the template was built from a different architecture."""
    template.load_state_dict(fold_bn(model).state_dict(), strict=True)
    return template


@torch.inference_mode()
def eval_model(compiled, eager, TX, TY, cfg, want_preds=False):
    """Confusion matrix of one folded model over the whole resident test set.

    Counts stay on the device: a per-batch .item() would sync ~650 times a pass, and the
    argmax of a row of NaN is 0 -- a perfectly ordinary class index -- so non-finite logits
    are counted explicitly instead of trusted. The tail batch runs eagerly (different shape
    would recompile the graph), same weights, same math."""
    C, EB, n = cfg["num_classes"], cfg["eval_batch"], TX.shape[0]
    dev = TX.device
    cm = torch.zeros(C * C, dtype=torch.long, device=dev)
    nonfin = torch.zeros((), dtype=torch.long, device=dev)
    preds = torch.empty(n, dtype=torch.uint8, device=dev) if want_preds else None
    ac = torch.autocast("cuda", dtype=torch.float16) if dev.type == "cuda" \
        else torch.autocast("cpu", enabled=False)
    for i in range(0, n, EB):
        j = min(i + EB, n)
        m = compiled if j - i == EB else eager
        # The previous batch's logits are still referenced by `z` when the next replay
        # starts; without an explicit step boundary CUDA-graph trees treat the call as
        # part of the same iteration and record a NEW graph node instead of replaying
        # (measured locally: 15k rows/s "compiled" vs 242k eager).
        if dev.type == "cuda":
            torch.compiler.cudagraph_mark_step_begin()
        with ac:
            z = m(TX[i:j].float())
        nonfin += (~torch.isfinite(z)).sum()
        p = z.argmax(1)
        cm += torch.bincount(TY[i:j].long() * C + p, minlength=C * C)
        if want_preds:
            preds[i:j] = p.to(torch.uint8)
    return cm.view(C, C), int(nonfin.item()), preds


In [ ]:
%%writefile /kaggle/working/proj/driver.py
"""One persistent worker per GPU, spawned once for the whole run.

Both GPUs hold the entire partition and the whole test set, so any client can train on
whichever GPU is free (longest-first dispatch), and the evaluation of the ONE global model
splits the test rows between the two GPUs by range, each worker returning a partial
confusion matrix that the driver sums. Each worker also holds a resident copy of the
server's aggregate theta^t, refreshed after every round, so a train task carries only a
client id, its row span and the round number.

Every client starts the round from theta^t (Algorithm 5 line 4), trains one local epoch,
and reports its weights; the server averages all N of them with weight 1/N (Eq. 9).
Aggregation is re-sorted by client id so float addition order never depends on which
worker finished first; every client re-seeds the default generator from
(seed, round, client) so its update does not depend on the schedule either. Different
clients are different local optimizations of the same model: they never form a process
group.
"""
import json, math, shutil, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.multiprocessing as mp

from proj.model import build_model, n_params, N_PARAMS_FULL
from proj.lwfednids import (layout, flatten, unflatten_into, client_update, aggregate,
                            amp, lr_at, make_optimizer, ce_loss, ACC_KEYS)
from proj.evaluate import fold_bn, load_folded, eval_model
from proj.metrics import metrics_from_confusion, per_class_from_confusion, METRIC_KEYS
from proj import ckpt as C


def model_mib(model):
    """What one model costs on the wire: every state_dict tensor in fp32."""
    return sum(v.numel() for v in model.state_dict().values()) * 4 / 2**20


def _resident(path, dev, chunk=1 << 22):
    """mmap -> GPU in chunks. A whole-array np.ascontiguousarray would materialise 5.7 GB
    of train features in host RAM per worker before the copy, and hands torch a read-only
    array. Chunking bounds the host side to `chunk` rows."""
    a = np.load(path, mmap_mode="r")
    t = torch.empty(tuple(a.shape), dtype=torch.from_numpy(np.array(a[:1])).dtype,
                    device=dev)
    for i in range(0, len(a), chunk):
        t[i:i + chunk] = torch.from_numpy(np.array(a[i:i + chunk]))
    return t


def _verdict(ref_z, got_z, ref_g, got_g, n_rows, label):
    """Decisive-row argmax agreement plus bounded logit/gradient deltas. Counts are
    integers: torch.mean on CUDA returns 0.99999994 for a perfect match."""
    dz = (got_z - ref_z).abs().max().item()
    gn = ref_g.norm().item()
    dg = (got_g - ref_g).norm().item() / (gn + 1e-12)
    flip_all = int((got_z.argmax(1) != ref_z.argmax(1)).sum())
    top2 = ref_z.topk(2, dim=1).values
    decisive = (top2[:, 0] - top2[:, 1]) > max(10 * dz, 1e-3)
    n_dec = int(decisive.sum())
    flip_dec = int((got_z.argmax(1) != ref_z.argmax(1))[decisive].sum())
    # `flip_dec == 0` over an EMPTY decisive set says nothing at all.
    if n_dec < n_rows // 10:
        raise RuntimeError(f"{label}: cannot certify, only {n_dec} of {n_rows} rows have "
                           f"a margin above {max(10 * dz, 1e-3):.2e}")
    if flip_dec or not math.isfinite(dz) or not math.isfinite(dg) or dz > 5e-2 or dg > 5e-2:
        raise RuntimeError(f"{label}: mismatch dlogit={dz} dgrad_rel={dg} "
                           f"flips {flip_dec}/{n_dec} decisive, {flip_all} of all")
    return f"max|dlogit|={dz:.2e} rel|dgrad|={dg:.2e} flips {flip_all}/{n_rows} ({flip_dec}/{n_dec} decisive)"


def _compile_train(Se, cfg, dev, xb, yb, rank=0):
    """reduce-overhead captures forward+backward into a CUDA graph. torch.compile is lazy,
    so a try around the call catches nothing -- run the production step and compare
    against eager from the SAME state, with Dropout off (Inductor functionalises RNG, so
    the two masks can never coincide). Warm-up captures the graphs at the production p,
    so restoring p reuses those entries."""
    if not cfg["compile"]:
        return Se
    drops = [m for m in Se.modules() if isinstance(m, nn.Dropout)]
    keep = [m.p for m in drops]
    snap = {k: v.detach().clone() for k, v in Se.state_dict().items()}
    rng = torch.get_rng_state()
    crng = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None

    def restore():
        with torch.no_grad():
            sd = Se.state_dict()
            for k, v in snap.items():
                sd[k].copy_(v)                      # copy_ keeps addresses -> graph stays valid
        torch.set_rng_state(rng)
        if crng is not None: torch.cuda.set_rng_state_all(crng)
        Se.zero_grad(set_to_none=True)
        Se.train()

    def probe(Sm):
        """The production step: forward under autocast, CE in fp32, one backward. Returns
        the logits and the flat gradient."""
        restore()
        if xb.is_cuda:
            torch.compiler.cudagraph_mark_step_begin()
        with amp(cfg):
            z = Sm(xb)
        z = z.float()
        loss = ce_loss(z, yb)
        loss.backward()
        g = torch.cat([p.grad.reshape(-1).float().clone() for p in Se.parameters()])
        Se.zero_grad(set_to_none=True)
        return z.clone(), g

    try:
        Sc = torch.compile(Se, mode="reduce-overhead")     # CUDA graphs: the 2-3x on T4
        for _ in range(3):                                  # warm up + capture at production p
            probe(Sc)
        for m in drops: m.p = 0.0
        try:
            ref = probe(Se)
            got = probe(Sc)
        finally:
            for m, p_ in zip(drops, keep): m.p = p_
        restore()
        s1 = _verdict(ref[0], got[0], ref[1], got[1], xb.shape[0], "train")
        print(f"[rank{rank}] compile OK | {s1}", flush=True)
        return Sc
    except Exception as e:                        # sm_75 Triton is the documented risk
        for m, p_ in zip(drops, keep): m.p = p_   # never leave the model with dropout off
        restore()
        print(f"[rank{rank}] compile DISABLED -> eager: {e}", flush=True)
        return Se


def _compile_eval(Te, cfg, dev, xt, rank=0):
    """The folded eval template, compiled at the fixed eval batch. Certified against the
    eager folded template on real test rows: fp16 cannot be bit-equal, so the criterion is
    the decisive-row argmax rule with a delta ceiling."""
    if not cfg["compile"]:
        return Te
    try:
        Tc = torch.compile(Te, mode="reduce-overhead")
        with torch.inference_mode():
            for _ in range(3):
                with amp(cfg):
                    Tc(xt).float()
            with amp(cfg):
                ref = Te(xt).float().clone()
                got = Tc(xt).float().clone()
        z = torch.zeros(1, device=dev)
        s = _verdict(ref, got, z, z, xt.shape[0], "eval")
        print(f"[rank{rank}] eval compile OK | {s}", flush=True)
        return Tc
    except Exception as e:
        print(f"[rank{rank}] eval compile DISABLED -> eager: {e}", flush=True)
        return Te


def eval_bounds(n, world_size):
    """Fixed row ranges of the test set, one per worker. A function of (n, W) only, so the
    same worker always scores the same rows and a 1-worker run sums to the same matrix."""
    return [(i * n // world_size, (i + 1) * n // world_size) for i in range(world_size)]


def worker(rank, cfg, task_q, res_q):
    """Wrapper: a worker that dies silently leaves the parent with only an exit code, and
    the real error is always in the CHILD traceback, not the spawn wrapper."""
    try:
        _worker(rank, cfg, task_q, res_q)
    except Exception:
        import traceback
        res_q.put(("error", rank, traceback.format_exc()))
        raise


def _worker(rank, cfg, task_q, res_q):
    cuda = cfg.get("device", "cuda") == "cuda"
    dev = torch.device(f"cuda:{rank}" if cuda else "cpu")
    if cuda:
        torch.cuda.set_device(dev)
        torch.backends.cudnn.benchmark = True
    # The train module and the folded eval template share ONE code object
    # (DAGSNet.forward), and Dynamo caches per code object: train/eval mode, dropout
    # on/off for the gate, no_grad vs grad, and two batch shapes can pass the default
    # recompile limit of 8. Past the limit Dynamo runs the new variant EAGERLY without
    # raising, so the eval template would silently lose CUDA graphs.
    for name in ("recompile_limit", "cache_size_limit"):
        if hasattr(torch._dynamo.config, name):
            setattr(torch._dynamo.config, name, 64)
    torch.manual_seed(cfg["seed"] + rank)
    cache = Path(cfg["cache"])
    X = _resident(cache / "train_X.f16.npy", dev)
    Y = _resident(cache / "train_y.u8.npy", dev)
    TX = _resident(cache / "test_X.f16.npy", dev)
    TY = _resident(cache / "test_y.u8.npy", dev)

    Se = build_model(cfg).to(dev).train()                # theta_j (per task)
    fk, ik, nP = layout(Se)
    assert nP == cfg["n_params"], (nP, cfg["n_params"])
    B = cfg["batch"]
    # Probe on real rows: random N(0,1) has none of the heavy tails of the z-scored
    # features, and a kernel that is wrong only at large magnitude would pass on noise.
    Sc = _compile_train(Se, cfg, dev, X[:B].float(), Y[:B].long(), rank)
    Te = fold_bn(build_model(cfg).to(dev))           # folded STRUCTURE; weights per round
    Tc = _compile_eval(Te, cfg, dev, TX[:cfg["eval_batch"]].float(), rank)
    scaler_probe = torch.amp.GradScaler("cuda", enabled=cuda)
    if cuda:
        scaler_probe.scale(torch.zeros(1, device=dev))  # force _scale to exist
        assert scaler_probe._scale is not None, "GradScaler._scale gone; skips would read as 0"
    res_q.put(("ready", rank, "eager" if Sc is Se else "compiled",
               "eager" if Tc is Te else "compiled"))

    GW = GI = None
    while True:
        task = task_q.get()
        kind = task[0]
        if kind == "stop":
            return
        if kind == "backend":
            # Both ranks gate independently, so one can compile and the other fall back.
            # A round whose clients were trained on two different backends is not a round
            # anyone can reproduce; the driver forces the lower common denominator.
            if task[1] == "eager": Sc = Se
            if task[2] == "eager": Tc = Te
            res_q.put(("backend_ok", rank, "eager" if Sc is Se else "compiled",
                       "eager" if Tc is Te else "compiled"))
            continue
        # Payloads cross the process boundary as numpy arrays: a torch tensor on a
        # multiprocessing queue is shared through /dev/shm, which a container may cap.
        if kind == "set_global":
            gw, gi = torch.from_numpy(task[1]).to(dev), torch.from_numpy(task[2]).to(dev)
            if GW is None:
                GW, GI = gw, gi
            else:
                GW.copy_(gw); GI.copy_(gi)
            res_q.put(("set_ok", rank))
            continue
        if kind == "eval":
            lo, hi, want_preds = task[1], task[2], task[3]
            t0 = time.monotonic()
            if cuda: torch.cuda.reset_peak_memory_stats(dev)
            unflatten_into(Se, GW, GI, fk, ik)
            load_folded(Te, Se)
            cm, nf, p = eval_model(Tc, Te, TX[lo:hi], TY[lo:hi], cfg, want_preds)
            res_q.put(("eval", rank, lo, hi, cm.cpu().numpy(), nf,
                       p.cpu().numpy() if want_preds else None,
                       (torch.cuda.max_memory_allocated(dev) / 2**30) if cuda else 0.0,
                       time.monotonic() - t0))
            continue
        if kind == "train":
            cid, lo, hi, rnd = task[1], task[2], task[3], task[4]
            t0 = time.monotonic()
            if cuda: torch.cuda.reset_peak_memory_stats(dev)
            unflatten_into(Se, GW, GI, fk, ik)                # Algorithm 5 line 4
            # AdamW re-created per client per round (owner's decision): every client
            # receives fresh weights from the server, and carrying moments across that
            # discontinuity would apply the previous model's curvature to a new one.
            lr = lr_at(cfg, rnd)
            opt = make_optimizer(Se, lr, cfg, fused=cuda)
            scaler = torch.amp.GradScaler("cuda", enabled=cuda)
            if cuda:
                scaler.scale(torch.zeros(1, device=dev))
            # Every stochastic input to this client derives from (seed, round, client):
            # the default generator drives Dropout, `g` drives the shuffles.
            s = cfg["seed"] * 1_000_003 + rnd * 10_007 + cid
            torch.manual_seed(s)
            g = torch.Generator(device=dev); g.manual_seed(s)
            acc, n = client_update(Sc, Se, opt, scaler, X, Y, lo, hi, cfg, g)
            sv, si = flatten(Se, fk, ik)
            a = acc.cpu().tolist()
            sk = int(a[len(ACC_KEYS)]); ap = n - sk                # NOT max(1, .): 0 must stay 0
            d_ = max(1, ap)
            st = {"cid": cid, "n_k": hi - lo, "rank": rank, "seed": s,
                  "lr": lr, "steps": n, "applied": ap, "skipped": sk,
                  "nonfinite": int(a[len(ACC_KEYS) + 1]),
                  "sec": time.monotonic() - t0,
                  "vram_gb": (torch.cuda.max_memory_allocated(dev) / 2**30 if cuda else 0.0)}
            st.update({k: a[j] / d_ for j, k in enumerate(ACC_KEYS)})   # means over applied steps
            res_q.put(("train", cid, hi - lo, sv.cpu().numpy(), si.cpu().numpy(), st))


def check_updates(rnd, results, stats, n_clients, max_skips=None):
    """Every reason a round must not be aggregated, in one pure function so it can be
    tested without two GPUs and a spawned worker.

    Skipped steps are NOT a failure: each client starts a fresh GradScaler at 2**16 and
    spends a few steps calibrating. What must be rejected is a client that applied no
    step, one that skipped far more than calibration explains, one whose APPLIED steps
    carried a non-finite gradient, and non-finite weights."""
    if sorted(stats) != list(range(n_clients)):
        raise RuntimeError(f"round {rnd}: reported {sorted(stats)}, expected 0..{n_clients - 1}")
    bad = [cid for cid, _, sv, _, _ in results if not np.isfinite(sv).all()]
    if bad:
        raise RuntimeError(f"round {rnd}: non-finite weights from clients {bad}")
    for c in sorted(stats):
        st = stats[c]
        if st["applied"] + st["skipped"] != st["steps"]:
            raise RuntimeError(f"round {rnd}: client {c} applied+skipped != steps")
    dead = [c for c in sorted(stats) if stats[c]["applied"] == 0]
    if dead:
        raise RuntimeError(f"round {rnd}: clients {dead} applied zero steps; "
                           "they would contribute unchanged weights")
    diverged = [c for c in sorted(stats) if stats[c]["nonfinite"]]
    if diverged:
        raise RuntimeError(f"round {rnd}: clients {diverged} APPLIED a step whose "
                           "gradient was not finite")
    if max_skips is not None:
        over = [c for c in sorted(stats) if stats[c]["skipped"] > max_skips]
        if over:
            raise RuntimeError(
                f"round {rnd}: clients {over} skipped more steps than the warm-up "
                f"budget ({max_skips}): "
                + ", ".join(f"{c}={stats[c]['skipped']}" for c in over))


def _collect(res_q, procs, n, timeout=7200):
    """A worker killed by the OS puts nothing on the queue. Poll in short slices and check
    liveness between them, or an OOM kill becomes a multi-hour hang."""
    out, deadline = [], time.time() + timeout
    while len(out) < n:
        try:
            msg = res_q.get(timeout=2.0)
            if msg[0] == "error":
                raise RuntimeError(f"worker {msg[1]} raised:\n{msg[2]}")
            out.append(msg)
        except RuntimeError:
            raise
        except Exception:
            for p in procs:
                if not p.is_alive() and p.exitcode not in (0, None):
                    raise RuntimeError(f"worker {p.pid} died, exitcode {p.exitcode} "
                                       f"(negative = signal; -9 is the OOM killer)")
            if time.time() > deadline:
                raise RuntimeError(f"timed out waiting for {n - len(out)} results")
    return out


def _shutdown(procs, task_qs):
    for q in task_qs:
        try: q.put(("stop",))
        except Exception: pass
    for p in procs:
        p.join(timeout=60)
        if p.is_alive():
            p.terminate(); p.join(timeout=10)


def _broadcast(task_qs, res_q, procs, msg):
    for q in task_qs: q.put(msg)
    return _collect(res_q, procs, len(task_qs))


def run(cfg, spans, class_names, wandb_run=None, t_origin=None):
    """t_origin is a time.monotonic() reading from when the SESSION started, not from when
    this call did. Worker spawn, the resident copy and compilation are minutes the 12 h cap
    charges for, and a deadline that started here would happily begin a round the session
    cannot finish."""
    if bool(cfg.get("prune_plan")) != (float(cfg["sparsity"]) > 0):
        raise SystemExit(f"cfg['sparsity'] = {cfg['sparsity']} but prune_plan is "
                         f"{'present' if cfg.get('prune_plan') else 'absent'}")
    t_start = t_origin if t_origin is not None else time.monotonic()
    mp.set_start_method("spawn", force=True)
    ctx = mp.get_context("spawn")
    task_qs = [ctx.Queue() for _ in range(cfg["world_size"])]
    res_q = ctx.Queue()
    procs = [ctx.Process(target=worker, args=(r, cfg, task_qs[r], res_q), daemon=True)
             for r in range(cfg["world_size"])]
    try:
        for p in procs: p.start()
        ready = _collect(res_q, procs, cfg["world_size"], timeout=3600)
        bt = {m[1]: m[2] for m in ready}; be = {m[1]: m[3] for m in ready}
        if len(set(bt.values())) > 1 or len(set(be.values())) > 1:
            print(f"[driver] ranks disagree on backend train={bt} eval={be}; forcing eager",
                  flush=True)
            force = ("backend", "eager" if len(set(bt.values())) > 1 else "keep",
                     "eager" if len(set(be.values())) > 1 else "keep")
            acks = _broadcast(task_qs, res_q, procs, force)
            bt = {m[1]: m[2] for m in acks}; be = {m[1]: m[3] for m in acks}
        cfg["backend"] = sorted(set(bt.values()))[0]
        cfg["backend_eval"] = sorted(set(be.values()))[0]
        startup = time.monotonic() - t_start
        print(f"[driver] {cfg['world_size']} workers ready: train {cfg['backend']}, "
              f"eval {cfg['backend_eval']} ({startup:.0f}s into the session)", flush=True)
        cfg["startup_seconds"] = startup
        # Push the effective backend somewhere READABLE WHILE THE RUN IS ALIVE: a running
        # Kaggle kernel's stdout cannot be downloaded.
        if wandb_run is not None:
            try:
                wandb_run.config.update({"backend": cfg["backend"],
                                         "backend_eval": cfg["backend_eval"],
                                         "startup_seconds": round(startup, 1)},
                                        allow_val_change=True)
                wandb_run.summary["backend"] = cfg["backend"]
                wandb_run.summary["backend_eval"] = cfg["backend_eval"]
            except Exception as e:
                print(f"[driver] could not publish backend to W&B: {e}", flush=True)
        return _rounds(cfg, spans, class_names, wandb_run, t_start, procs, task_qs, res_q)
    finally:
        # Without this a driver-side exception leaves two processes holding both GPUs, and
        # the next cell in the notebook fails with a CUDA OOM that names nothing.
        _shutdown(procs, task_qs)


def _stats(values):
    v = np.asarray(values, dtype=np.float64)
    return float(v.mean()), float(v.std()), float(v.min()), float(v.max())


def _rounds(cfg, spans, class_names, wandb_run, t_start, procs, task_qs, res_q):
    d = C.run_dir(cfg["run_name"])
    N, W = cfg["n_clients"], cfg["world_size"]
    # Seed BEFORE building: theta_0 is the seeded unpruned DAGSNet, and build_model slices
    # it by the plan the server computed from that same theta_0 (Eq. 8: theta' = M ⊙ theta_0).
    torch.manual_seed(cfg["seed"])
    M0 = build_model(cfg)
    if n_params(M0) != cfg["n_params"]:
        raise SystemExit(f"model has {n_params(M0):,} parameters, cfg says {cfg['n_params']:,}")
    fk, ik, nP = layout(M0)
    GW, GI = flatten(M0, fk, ik)
    mib = model_mib(M0)

    start = 1
    last = C.resolve_resume(cfg["run_name"], cfg)
    if last is not None:
        G, ck = C.load_weights(d / "weights" / f"round_{last:03d}.pt", build_model,
                               cfg["n_params"])
        GW, GI = flatten(G, fk, ik)
        start = last + 1
        print(f"[driver] resumed at round {start}")
    elif cfg.get("require_resume"):
        raise SystemExit("require_resume set and no checkpoint found")
    if start > cfg["rounds"]:
        print(f"[driver] nothing to do: {last} rounds already complete")
        return []
    _broadcast(task_qs, res_q, procs, ("set_global", GW.numpy(), GI.numpy()))

    n_test = cfg["n_test"]
    bounds = eval_bounds(n_test, W)
    preds_rounds = set(cfg.get("preds_rounds", [cfg["rounds"]]))
    hist = []
    reserve = cfg.get("finalize_reserve_seconds", 600)
    elapsed = time.monotonic() - t_start
    if elapsed + reserve >= cfg["max_seconds"]:
        print(f"[driver] no round started: {elapsed/3600:.2f} h of the "
              f"{cfg['max_seconds']/3600:.2f} h budget is already gone", flush=True)
        return hist

    for rnd in range(start, cfg["rounds"] + 1):
        t0 = time.monotonic()
        # Algorithm 6 lines 6-8: every client trains this round. Longest-first bounds the
        # idle tail: sending the biggest client last strands a GPU.
        order = sorted(range(N), key=lambda c: spans[c][1] - spans[c][0], reverse=True)
        pending, nxt, results = {}, 0, []
        for r in range(W):                                  # prime both GPUs
            if nxt < len(order):
                c = order[nxt]; nxt += 1
                task_qs[r].put(("train", c, *spans[c], rnd)); pending[r] = c
        while len(results) < len(order):
            msg = _collect(res_q, procs, 1)[0]
            assert msg[0] == "train", msg[0]
            results.append(msg[1:])
            r = next(k for k, v in pending.items() if v == msg[1])
            if nxt < len(order):
                c = order[nxt]; nxt += 1
                task_qs[r].put(("train", c, *spans[c], rnd)); pending[r] = c
            else:
                pending.pop(r)
        t_train = time.monotonic() - t0

        results.sort(key=lambda t: t[0])                    # NOT completion order
        stats = {cid: s for cid, _, _, _, s in results}
        check_updates(rnd, results, stats, N, cfg.get("max_skips_per_client"))
        # Eq. (9): the unweighted mean over ALL N clients.
        GW, GI = aggregate([(cid, torch.from_numpy(sv), torch.from_numpy(si))
                            for cid, _, sv, si, _ in results], N)
        _broadcast(task_qs, res_q, procs, ("set_global", GW.numpy(), GI.numpy()))

        # ---- evaluate theta^t on the full test set, rows split between the workers.
        want_preds = rnd in preds_rounds
        t1 = time.monotonic()
        for r in range(W):
            task_qs[r].put(("eval", *bounds[r], want_preds))
        ev = sorted(_collect(res_q, procs, W), key=lambda e: e[2])      # by row range
        t_eval = time.monotonic() - t1
        assert [(e[2], e[3]) for e in ev] == bounds, [(e[2], e[3]) for e in ev]
        nf_total = sum(e[5] for e in ev)
        if nf_total:
            raise RuntimeError(f"round {rnd}: {nf_total} non-finite test logits; argmax "
                               "would have turned them into ordinary class labels")
        cm = np.sum([e[4] for e in ev], axis=0)
        assert cm.sum() == n_test, f"confusion matrix has {cm.sum()} of {n_test} test rows"
        preds = np.concatenate([e[6] for e in ev]) if want_preds else None
        gm = metrics_from_confusion(cm)

        row = {"round": rnd, "lr": lr_at(cfg, rnd), "n_params": cfg["n_params"],
               "sparsity": cfg["sparsity"]}
        row.update(gm)                                        # the 10 metrics of theta^t
        # *_client_mean is the unweighted mean ACROSS CLIENTS of each client's mean over
        # its applied steps -- not the mean over training samples. std/min/max alongside.
        for k in ACC_KEYS:
            mean, std, lo, hi = _stats([s[k] for s in stats.values()])
            row[f"{k}_client_mean"] = mean
            row[f"{k}_client_std"], row[f"{k}_client_min"], row[f"{k}_client_max"] = std, lo, hi
        row["sec_client_mean"] = float(np.mean([s["sec"] for s in stats.values()]))
        row.update({
            "steps": int(sum(s["steps"] for s in stats.values())),
            "applied": int(sum(s["applied"] for s in stats.values())),
            "skipped": int(sum(s["skipped"] for s in stats.values())),
            # What the protocol transmits this round: N models down at the start of the
            # round and N up at the end, each the (pruned) model in fp32.
            "model_mib": mib, "comm_mib": 2 * N * mib,
            "train_sec": t_train, "eval_sec": t_eval,
            "vram_train_gb": max(s["vram_gb"] for s in stats.values()),
            "vram_eval_gb": max(e[7] for e in ev),
            "backend": cfg.get("backend", "?"), "seconds": 0.0})
        clients = [stats[c] for c in sorted(stats)]

        # ---- commit. Marker absolutely last.
        unflatten_into(M0, GW, GI, fk, ik)
        C.save_round_weights(C.cpu_sd(M0), rnd, cfg, gm, d)
        C.atomic_np_save(d / "confusion" / f"round_{rnd:03d}.npy", cm)
        if want_preds:
            C.atomic_np_save(d / "preds" / f"round_{rnd:03d}.u8.npy", preds)
        (d / "logs" / f"round_{rnd:03d}.json").write_text(
            json.dumps({"round": rnd, "clients": clients}, indent=1))
        # `seconds` BEFORE the W&B call and the JSON: the budget below compares absolute
        # session elapsed, so the commit tail lands in the next round's elapsed and the
        # finalize reserve covers the last one.
        row["seconds"] = time.monotonic() - t0
        if wandb_run is not None:
            # W&B is a monitor, never a dependency: in a resumed session the service has
            # died at startup before and every later log call went nowhere. The round's
            # artifacts below are the record; a W&B failure must not touch them.
            try:
                wandb_run.log({k: v for k, v in row.items() if k not in ("round", "backend")},
                              step=rnd)
            except Exception as e:
                print(f"[driver] W&B log failed at round {rnd}: {e}", flush=True)
        (d / "metrics" / f"round_{rnd:03d}.json").write_text(json.dumps(
            {**row, "per_class": per_class_from_confusion(cm, class_names),
             "clients": clients}, indent=1))
        C.append_history(d, row, [{"round": rnd, **c} for c in clients])
        C.mark_complete(d, rnd)
        hist.append(row)
        print(f"[r{rnd:03d}] f1_macro={row['f1_macro']:.6f} acc={row['accuracy']:.6f} "
              f"f1_w={row['f1_weighted']:.6f} | lr={row['lr']:.2e} "
              f"loss={row['loss_client_mean']:.4f}±{row['loss_client_std']:.4f} "
              f"gnorm={row['gnorm_client_mean']:.3f} skip={row['skipped']}/{row['steps']} "
              f"| train={t_train:.0f}s eval={t_eval:.0f}s vram={row['vram_train_gb']:.2f}G "
              f"{row['seconds']:.1f}s | session {(time.monotonic()-t_start)/3600:.2f}h",
              flush=True)

        worst = max(h["seconds"] for h in hist)
        if (time.monotonic() - t_start) + worst * 1.15 + reserve > cfg["max_seconds"]:
            print(f"[driver] stopping after round {rnd}: the next round plus a "
                  f"{reserve/60:.0f} min finalize reserve would exceed the session budget "
                  f"({cfg['max_seconds']/3600:.2f} h)", flush=True)
            break

    return hist


def write_manifest(cfg, class_names, spans, y_true_src=None, extra=None):
    """Everything needed to say what these numbers are, written once, next to them.

    Also the SECOND data gate: `content_id` is computed after the decode from the row
    counts and the class histogram and compared against what the resumed checkpoint was
    trained on. Continuing on top of different data is not a warning."""
    d = C.run_dir(cfg["run_name"])
    mf = d / "reports" / "manifest.json"
    m = {"fingerprint": C.fingerprint(cfg),
         "cfg": {k: v for k, v in cfg.items()},
         "class_names": list(class_names),
         "n_clients": len(spans), "n_train": sum(h - l for l, h in spans.values()),
         "client_rows": {str(c): spans[c][1] - spans[c][0] for c in sorted(spans)},
         "n_params": cfg["n_params"], "n_params_full": N_PARAMS_FULL,
         "sparsity": cfg["sparsity"], "plan_id": cfg["plan_id"],
         "torch": torch.__version__, "cuda": torch.version.cuda,
         "written": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    if extra: m.update(extra)

    # A handoff bundle carries the ids of the data its rounds were trained on; a resumed
    # session has no earlier manifest, so this is where that gate fires for it.
    old = C.read_handoff(d) if not mf.is_file() else None
    if old is not None:
        for k in ("content_id", "data_id"):
            a, b = old.get(k), m.get(k)
            if a is not None and b is not None and a != b:
                raise RuntimeError(
                    f"{k} changed: the handoff's rounds were trained on {a}, the data "
                    f"mounted now is {b}. Resuming across that is not a continuation.")
    if mf.is_file():
        old = json.loads(mf.read_text())
        for k in ("content_id", "data_id"):
            a, b = old.get(k), m.get(k)
            if a is not None and b is not None and a != b:
                raise RuntimeError(
                    f"{k} changed: this run's checkpoints were trained on {a}, the data "
                    f"mounted now is {b}. Resuming across that is not a continuation.")
        m["sessions"] = int(old.get("sessions", 1)) + 1
        m["first_written"] = old.get("first_written", old.get("written"))
    else:
        m["sessions"], m["first_written"] = 1, m["written"]

    # y_true travels with the run: a downloaded run directory must be able to check its
    # own predictions against its own confusion matrices.
    if y_true_src is not None:
        dst = d / "reports" / "y_true.u8.npy"
        if not dst.is_file():
            shutil.copyfile(y_true_src, dst)
        m["y_true"] = "reports/y_true.u8.npy"
    mf.write_text(json.dumps(m, indent=2))
    return mf


In [ ]:
%%writefile /kaggle/working/proj/verify.py
"""Re-derive every published number from the artifacts on disk.

Nothing here trusts a number because it was printed once. Per round, the 10 metrics and
the per-class block are recomputed from the global model's confusion matrix; the means /
std / min / max over clients of the training statistics are recomputed from the per-client
log; predictions, where stored, rebuild the confusion matrix; the client log is checked
against the step arithmetic and the schedule it claims; the weights file must rebuild the
(pruned) model at the parameter count its own cfg declares; and all of it is compared
against the copies in the weights file, the metrics json, history.csv and clients.csv.

Every check exists because its absence lets a specific tampered fixture pass: a deleted
history row, a duplicated one, a metric set to NaN (`abs(nan) > tol` is False), a
per-class F1 of 999, deleted logs, a client missing from the log, a metric overwritten in
one of its four copies, a resume file overwritten with garbage, a config claiming 999
test rows, a plan whose parameter count is not the one the weights were trained at.
"""
import csv, json, math
from pathlib import Path
import numpy as np
import torch

from proj.metrics import metrics_from_confusion, per_class_from_confusion, METRIC_KEYS
from proj.lwfednids import expected_steps, lr_at, ACC_KEYS
from proj import ckpt as C

TOL = 1e-12          # both sides come from the same float64 code path on the same counts
CSV_TOL = 1e-9       # history.csv round-trips through str()
CLIENT_INT = ("cid", "n_k", "steps", "applied", "skipped", "nonfinite")
CLIENT_FLOAT = ("lr",) + ACC_KEYS


def _finite(x):
    try: return math.isfinite(float(x))
    except (TypeError, ValueError): return False


def _read_csv(p):
    with open(p) as f:
        return list(csv.DictReader(f))


def verify_run(run_dir, cfg=None, build_model=None, expect_params=None, y_true_path=None,
               require_rounds=None, full=True):
    """Returns (ok, lines). full=True is the acceptance mode: client logs must be present
    for every round and predictions for every round that claims them."""
    d = Path(run_dir)
    fp = C.fingerprint(cfg) if cfg is not None else None
    last = C.last_complete_round(d, fp) or 0
    # A tree that starts past round 1 is a session resumed from a handoff bundle: rounds
    # before `first` are attested by reports/handoff.json (checked inside round_ok) and are
    # re-verified in full only after the sessions are merged. Nothing here is skipped for
    # the rounds that ARE on disk.
    first = C._first_round(d) if last else 1
    mode = "full" if full else "minimal"
    out = [f"run      : {d}", f"complete : rounds {first}..{last}   (mode: {mode})"
           + (f"   [handoff bundle: rounds 1..{first} attested by {C.HANDOFF}]" if first > 1 else "")]
    bad = []

    mf = d / "reports" / "manifest.json"
    man = json.loads(mf.read_text()) if mf.is_file() else None
    if man is None and full:
        bad.append("reports/manifest.json missing: the run does not describe itself")

    N = int(cfg["n_clients"]) if cfg else (int(man["n_clients"]) if man else None)
    n_test = int(cfg["n_test"]) if cfg and "n_test" in cfg else None
    want_rows = ({int(k): int(v) for k, v in man["client_rows"].items()}
                 if man and "client_rows" in man else None)
    want_params = int(cfg["n_params"]) if cfg else (int(man["n_params"]) if man else None)

    # ---- history.csv / clients.csv: exactly rounds first..last, once each
    hist, hp = {}, d / "history.csv"
    if hp.is_file():
        rows = _read_csv(hp)
        seen = [int(r["round"]) for r in rows]
        if len(seen) != len(set(seen)):
            bad.append(f"history.csv has duplicate rows for round(s) "
                       f"{sorted({r for r in seen if seen.count(r) > 1})}")
        if sorted(set(seen)) != list(range(first, last + 1)):
            bad.append(f"history.csv covers rounds {sorted(set(seen))}, expected {first}..{last}")
        hist = {int(r["round"]): r for r in rows}
    elif last:
        bad.append("history.csv missing")
    crows, cp = {}, d / "clients.csv"
    if cp.is_file():
        for r in _read_csv(cp):
            crows.setdefault(int(r["round"]), {})[int(r["cid"])] = r
    elif last:
        bad.append("clients.csv missing")

    y_true = None
    if y_true_path is not None and Path(y_true_path).is_file():
        y_true = np.load(y_true_path, mmap_mode="r")

    for r in range(first, last + 1):
        tag = f"round {r:03d}"
        cm = np.load(d / "confusion" / f"round_{r:03d}.npy")
        if cm.ndim != 2 or cm.shape[0] != cm.shape[1]:
            bad.append(f"{tag}: confusion is {cm.shape}, expected (C, C)"); continue
        if (cm < 0).any():
            bad.append(f"{tag}: negative counts in the confusion matrix")
        if cfg and cm.shape[0] != int(cfg["num_classes"]):
            bad.append(f"{tag}: {cm.shape[0]} classes, cfg says {cfg['num_classes']}")
        if n_test is None: n_test = int(cm.sum())
        if cm.sum() != n_test:
            bad.append(f"{tag}: confusion total {int(cm.sum())} != n_test {n_test}")

        js = json.loads((d / "metrics" / f"round_{r:03d}.json").read_text())
        ck = torch.load(d / "weights" / f"round_{r:03d}.pt", map_location="cpu",
                        weights_only=True, mmap=True)
        rs = torch.load(d / "resume" / f"round_{r:03d}.pt", map_location="cpu",
                        weights_only=True)
        if int(rs.get("round", -1)) != r:
            bad.append(f"{tag}: resume file claims round {rs.get('round')}")

        # ---- the global model's 10 metrics + per-class, in json, weights file, csv
        rec = metrics_from_confusion(cm)
        for k in METRIC_KEYS:
            for where, val, tol in (("json", js.get(k), TOL),
                                    ("history.csv", hist.get(r, {}).get(k), CSV_TOL),
                                    ("weights file", ck.get("metrics", {}).get(k), TOL)):
                if val is None:
                    bad.append(f"{tag}: {k} missing in {where}"); continue
                if not _finite(val):
                    bad.append(f"{tag}: {k} in {where} is not finite ({val!r})")
                elif abs(float(val) - rec[k]) > tol:
                    bad.append(f"{tag}: {k} in {where} = {val} != {rec[k]} recomputed")
        names = [e.get("class") for e in js.get("per_class", [])]
        if len(names) != cm.shape[0]:
            bad.append(f"{tag}: per-class block missing or wrong length")
        else:
            for got, want in zip(js["per_class"], per_class_from_confusion(cm, names)):
                for f in ("idx", "support"):
                    if int(got.get(f, -1)) != int(want[f]):
                        bad.append(f"{tag}: class {want['idx']} {f} {got.get(f)} != {want[f]}")
                for f in ("precision", "recall", "f1"):
                    v = got.get(f)
                    if not _finite(v) or abs(float(v) - want[f]) > TOL:
                        bad.append(f"{tag}: class {want['idx']} {f} {v!r} != {want[f]}")
        if cfg is not None:
            want_lr = lr_at(cfg, r)
            if not _finite(js.get("lr")) or abs(float(js["lr"]) - want_lr) > 1e-12:
                bad.append(f"{tag}: json lr {js.get('lr')!r} != schedule {want_lr}")
            if int(js.get("n_params", -1)) != want_params:
                bad.append(f"{tag}: json n_params {js.get('n_params')} != cfg {want_params}")
            if float(js.get("sparsity", -1)) != float(cfg["sparsity"]):
                bad.append(f"{tag}: json sparsity {js.get('sparsity')} != cfg {cfg['sparsity']}")
        if fp is not None and ck.get("fingerprint") != fp:
            bad.append(f"{tag}: weights fingerprint {ck.get('fingerprint')} != {fp}")
        if build_model is not None:
            try:
                C.load_weights(d / "weights" / f"round_{r:03d}.pt", build_model, expect_params)
            except Exception as e:
                bad.append(f"{tag}: weights do not rebuild the model: {e}")

        # ---- predictions tie the matrix back to model output, where stored
        pp = d / "preds" / f"round_{r:03d}.u8.npy"
        claims = cfg is not None and r in set(cfg.get("preds_rounds", [cfg["rounds"]]))
        if pp.is_file():
            yp = np.load(pp, mmap_mode="r")
            if yp.shape != (n_test,):
                bad.append(f"{tag}: predictions are {yp.shape}, expected ({n_test},)")
            elif y_true is not None:
                if len(y_true) != n_test:
                    bad.append(f"{tag}: y_true has {len(y_true)} rows, test has {n_test}")
                else:
                    k = cm.shape[0]
                    rebuilt = np.bincount(np.asarray(y_true, np.int64) * k
                                          + np.asarray(yp, np.int64),
                                          minlength=k * k).reshape(k, k)
                    if not (rebuilt == cm).all():
                        bad.append(f"{tag}: confusion != stored predictions")
            elif full:
                bad.append(f"{tag}: predictions present but no y_true to check them against")
        elif claims and full:
            bad.append(f"{tag}: cfg claims predictions for this round but none are stored")

        # ---- client logs: every client, step arithmetic and the schedule have to close;
        #      the json's copy, clients.csv and the round's client_* aggregates must agree
        lp = d / "logs" / f"round_{r:03d}.json"
        lg_clients = None
        if not lp.is_file():
            if full:
                bad.append(f"{tag}: no client log; participation is unattested")
        else:
            try: lg_clients = json.loads(lp.read_text()).get("clients", [])
            except Exception as e:
                bad.append(f"{tag}: client log unreadable: {e}"); lg_clients = []
        jc = js.get("clients")
        if jc is None:
            bad.append(f"{tag}: metrics json carries no per-client block")
        clients = lg_clients if lg_clients is not None else (jc or [])
        if lg_clients is not None and jc is not None and lg_clients != jc:
            bad.append(f"{tag}: logs/ and metrics/ disagree on the per-client block")
        got_ids = sorted(int(e["cid"]) for e in clients)
        if N is not None and got_ids != list(range(N)):
            bad.append(f"{tag}: log has clients {got_ids[:6]}..., expected all 0..{N - 1}")
        per = {k: [] for k in ACC_KEYS}
        for e in clients:
            c = e.get("cid")
            if e.get("applied", 0) + e.get("skipped", 0) != e.get("steps", -1):
                bad.append(f"{tag}: client {c} applied+skipped != steps")
            if e.get("applied", 0) <= 0:
                bad.append(f"{tag}: client {c} applied no step")
            if e.get("nonfinite", 0):
                bad.append(f"{tag}: client {c} applied a step with a non-finite gradient")
            if cfg and "n_k" in e:
                s = expected_steps(int(e["n_k"]), cfg)
                if int(e.get("steps", -1)) != s:
                    bad.append(f"{tag}: client {c} ran {e.get('steps')} steps, expected {s}")
            if want_rows is not None and c in want_rows and int(e.get("n_k", -1)) != want_rows[c]:
                bad.append(f"{tag}: client {c} trained on {e.get('n_k')} rows, "
                           f"manifest says {want_rows[c]}")
            # The rate the client actually used must be the schedule's value for this
            # round: a resumed session that planned a different horizon would otherwise
            # continue the run at a rate the fingerprint never saw.
            if cfg is not None:
                want_lr = lr_at(cfg, r)
                if not _finite(e.get("lr")) or abs(float(e["lr"]) - want_lr) > 1e-12:
                    bad.append(f"{tag}: client {c} trained at lr {e.get('lr')!r}, "
                               f"schedule says {want_lr}")
            for f in ACC_KEYS:
                if not _finite(e.get(f)):
                    bad.append(f"{tag}: client {c} {f} is not finite ({e.get(f)!r})")
                else:
                    per[f].append(float(e[f]))
            cv = crows.get(r, {}).get(int(c)) if crows else None
            if crows and cv is None:
                bad.append(f"{tag}: client {c} missing from clients.csv")
            elif cv is not None:
                for f in CLIENT_INT:
                    if str(cv.get(f)) != str(e.get(f)):
                        bad.append(f"{tag}: clients.csv client {c} {f} = {cv.get(f)!r} != {e.get(f)!r}")
                for f in CLIENT_FLOAT:
                    if not _finite(cv.get(f)) or abs(float(cv[f]) - float(e[f])) > CSV_TOL:
                        bad.append(f"{tag}: clients.csv client {c} {f} = {cv.get(f)!r} != {e[f]}")
        if clients and len(per[ACC_KEYS[0]]) == len(clients):
            for k in ACC_KEYS:
                v = np.asarray(per[k], dtype=np.float64)
                want = {f"{k}_client_mean": float(v.mean()), f"{k}_client_std": float(v.std()),
                        f"{k}_client_min": float(v.min()), f"{k}_client_max": float(v.max())}
                for kk, wv in want.items():
                    for where, val, tol in (("json", js.get(kk), TOL),
                                            ("history.csv", hist.get(r, {}).get(kk), CSV_TOL)):
                        if val is None:
                            bad.append(f"{tag}: {kk} missing in {where}"); continue
                        if not _finite(val):
                            bad.append(f"{tag}: {kk} in {where} is not finite ({val!r})")
                        elif abs(float(val) - wv) > tol:
                            bad.append(f"{tag}: {kk} in {where} = {val} != {wv} recomputed")
            for f in ("steps", "applied", "skipped"):
                want = sum(int(e.get(f, 0)) for e in clients)
                if int(js.get(f, -1)) != want:
                    bad.append(f"{tag}: json {f} {js.get(f)} != {want} summed over clients")

    n_preds = len(list((d / "preds").glob("round_*.u8.npy"))) if (d / "preds").is_dir() else 0
    n_logs = len(list((d / "logs").glob("round_*.json"))) if (d / "logs").is_dir() else 0
    out.append(f"artifacts: {n_preds} prediction files, {n_logs} client logs"
               + ("" if y_true is not None else "   (predictions NOT cross-checked: no y_true)"))
    if require_rounds is not None and last != require_rounds:
        bad.append(f"run is INCOMPLETE: {last} of {require_rounds} rounds")
    out += [f"  FAIL {b}" for b in bad] or ["  all artifact checks passed"]
    return not bad, out


In [ ]:
# ---- server initialization (§III-B-1): theta_0 from the seed, the mask from theta_0
# alone, theta' = M ⊙ theta_0 with the pruned channels removed. The plan is part of CFG
# from here on: proj/ckpt.py hashes it into the fingerprint (plan_id) and every weights
# file carries it, so build_model(cfg) rebuilds the pruned model without torch-pruning.
import json
from proj.prune import compute_plan
from proj.model import build_model, n_params
import torch
CFG["prune_plan"], PRUNE = compute_plan(CFG, verbose=True)
CFG["plan_id"], CFG["n_params"] = PRUNE["plan_id"], PRUNE["n_params"]
torch.manual_seed(CFG["seed"]); _m = build_model(CFG)
assert n_params(_m) == CFG["n_params"], (n_params(_m), CFG["n_params"])
print(f"{'layer':<26} {'type':<12} {'out':>10} {'in':>10}")
for _n, L in PRUNE["layers"].items():
    print(f"{_n:<26} {L['type']:<12} {L['out'][0]:>4}->{L['out'][1]:<4} "
          + (f"{L['in'][0]:>4}->{L['in'][1]:<4}" if L["in"] else ""))
print(f"pruned model: {CFG['n_params']:,} of {PRUNE['n_params_full']:,} params "
      f"({CFG['n_params']/PRUNE['n_params_full']:.2%}), MACs {PRUNE['macs_pruned']:,} of "
      f"{PRUNE['macs_full']:,} ({PRUNE['macs_pruned']/PRUNE['macs_full']:.2%}), plan {CFG['plan_id']}")
del _m


In [ ]:
import wandb
# BEGIN INLINE WANDB CREDENTIAL — owner-authorized private notebook
wandb.login(key=__import__("os").environ["WANDB_API_KEY"], relogin=True, verify=True)
# END INLINE WANDB CREDENTIAL
# W&B authenticates before dataset decode; inline key use was explicitly authorized by the owner.
run = wandb.init(project="lwfednids-veremi", name="lwfednids_20c_probe",
                 config={k: v for k, v in CFG.items() if k != "prune_plan"},
                 resume="allow", id="lwfednids_20c_probe")
print("W&B:", run.url)


In [ ]:
# Cheap identity work, then the resume gate, then the decode. A continuation push must
# die at the gate, not after a two-minute parquet pass.
import hashlib, json, time, numpy as np
from pathlib import Path
from proj import ckpt as C
from proj.data import (find_root, load_clients, load_test, assert_fp16_safe,
                       cache_ok)

FL_ROOT = find_root("train/client_id=000")
CEN_ROOT = find_root("upload/test")
TEST_ROOT = CEN_ROOT / "upload/test"
SCALER = json.loads((CEN_ROOT / "upload/scaler.json").read_text())["features"]
assert len(SCALER) == 66, f"scaler has {len(SCALER)} entries, expected 66"
FEATS = ['f_rcv_pos_noise_x', 'f_rcv_pos_noise_y', 'f_rcv_spd', 'f_rcv_spd_noise', 'f_rcv_acl', 'f_rcv_acl_noise', 'f_rcv_hed_noise', 'f_snd_pos_noise_x', 'f_snd_pos_noise_y', 'f_snd_spd', 'f_snd_spd_noise', 'f_snd_acl', 'f_snd_acl_noise', 'f_snd_hed_noise', 'f_snd_dist_road_edge', 'f_rcv_x_rel', 'f_rcv_y_rel', 'f_snd_x_rel', 'f_snd_y_rel', 'f_delay_s', 'f_dx', 'f_dy', 'f_dist', 'f_bearing_sin', 'f_bearing_cos', 'f_rcv_hed_sin', 'f_rcv_hed_cos', 'f_snd_hed_sin', 'f_snd_hed_cos', 'f_hed_diff_cos', 'f_rcv_vx', 'f_rcv_vy', 'f_snd_vx', 'f_snd_vy', 'f_rel_speed', 'f_closing_speed', 'f_spd_diff', 'f_rcv_noise_mag', 'f_snd_noise_mag', 'f_first_in_session', 'f_sess_idx', 'f_sess_dt', 'f_sess_dt_send', 'f_sess_dt_skew', 'f_sess_dpos', 'f_sess_implied_spd', 'f_sess_spd_residual', 'f_sess_dspd', 'f_sess_acl_residual', 'f_sess_dhed', 'f_sess_dmsgid', 'f_sess_ddist', 'f_sess_ddre', 'f_sess_pos_pred_err', 'f_alias_age_s', 'f_rx_rate_1s', 'f_rx_rate_5s', 'f_sender_rate_1s', 'f_sender_rate_5s', 'f_sender_share_5s', 'f_rcv_profile_normal', 'f_rcv_profile_cautious', 'f_rcv_profile_aggressive', 'f_snd_profile_normal', 'f_snd_profile_cautious', 'f_snd_profile_aggressive']
CLASS_NAMES = ['benign', 'accelerationMultiplication', 'constantPositionOffset', 'constantSpeedOffset', 'dataReplay', 'dosAttack', 'feignedBraking', 'positionMirroring', 'randomPositionOffset', 'randomSpeedOffset', 'reversedHeading', 'suddenConstantSpeed', 'suddenStop', 'timeDelayAttack', 'trafficCongestionSybil', 'zeroSpeedReport']

# What the fingerprint could not otherwise see: a permuted feature order, a re-fitted
# scaler or a different partition keep every shape identical.
CFG["data_id"] = hashlib.sha256(json.dumps({
    "features": FEATS, "classes": CLASS_NAMES, "n_clients": CFG["n_clients"],
    "scaler": [[SCALER[c]["mean"], SCALER[c]["std_used"]] for c in FEATS],
}, sort_keys=True).encode()).hexdigest()[:16]
print("FL root    :", FL_ROOT)
print("test root  :", TEST_ROOT)
print("data_id    :", CFG["data_id"])
print("fingerprint:", C.fingerprint(CFG))

last = C.resolve_resume(CFG["run_name"], CFG)
if CFG["require_resume"] and last is None:
    raise SystemExit("require_resume set but no VERIFIED checkpoint found — fix the "
                     "attachment. A marker without its artifacts does not count.")
print("resume from round", last)

cache = Path(CFG["cache"]); cache.mkdir(parents=True, exist_ok=True)
MF = cache / "manifest.json"
FILES = ("train_X.f16.npy", "train_y.u8.npy", "test_X.f16.npy", "test_y.u8.npy",
         "spans.json")
want = {"data_id": CFG["data_id"], "n_clients": CFG["n_clients"],
        "fl_root": str(FL_ROOT), "test_root": str(TEST_ROOT)}

t0 = time.time()
if cache_ok(cache, want, CFG["n_clients"]):
    print("prepack cache reusable (manifest matches and every file checks out)")
else:
    for f in FILES: (cache / f).unlink(missing_ok=True)
    MF.unlink(missing_ok=True)
    X, Y, spans = load_clients(FL_ROOT, FEATS, CFG["n_clients"])
    print(f"train {X.shape} max|x|={assert_fp16_safe(X,'train'):.1f}")
    np.save(cache / "train_X.f16.npy", X); np.save(cache / "train_y.u8.npy", Y)
    json.dump({str(k): v for k, v in spans.items()}, open(cache / "spans.json", "w"))
    del X, Y
    TX, TY = load_test(TEST_ROOT, FEATS, SCALER)
    print(f"test  {TX.shape} max|x|={assert_fp16_safe(TX,'test'):.1f}")
    np.save(cache / "test_X.f16.npy", TX); np.save(cache / "test_y.u8.npy", TY)
    del TX, TY
    MF.write_text(json.dumps(want))                       # cache marker: absolutely last
    assert cache_ok(cache, want, CFG["n_clients"]), "the cache just written does not validate"

spans = {int(k): tuple(v) for k, v in json.load(open(cache / "spans.json")).items()}
CFG["n_test"] = len(np.load(cache / "test_y.u8.npy", mmap_mode="r"))
n_train = sum(h - l for l, h in spans.values())
assert n_train == 43_045_415, f"train rows {n_train} != 43,045,415"
assert CFG["n_test"] == 10_761_343, f"test rows {CFG['n_test']}"
assert len(spans) == CFG["n_clients"], f"{len(spans)} spans for {CFG['n_clients']} clients"

# content_id reads the labels that are actually cached, hit or miss: the row counts and
# the class histogram change when the partition or the file contents change.
_ytr = np.load(cache / "train_y.u8.npy", mmap_mode="r")
_yte = np.load(cache / "test_y.u8.npy", mmap_mode="r")
assert len(_ytr) == n_train, f"train X/y disagree: {len(_ytr)} labels for {n_train} rows"
CFG["content_id"] = hashlib.sha256(json.dumps({
    "clients": [[c, spans[c][0], spans[c][1]] for c in sorted(spans)],
    "train_hist": np.bincount(np.asarray(_ytr), minlength=16).tolist(),
    "test_hist": np.bincount(np.asarray(_yte), minlength=16).tolist(),
}, sort_keys=True).encode()).hexdigest()[:16]
del _ytr, _yte
print("content_id :", CFG["content_id"])
print(f"prepack {time.time()-t0:.1f}s | {n_train:,} train / {CFG['n_test']:,} test rows")


In [ ]:
# ---- PROBE ONLY: measure what sm_86 cannot tell us about the T4 before the rounds run.
# Train: one client's epoch eager vs compiled on the PRUNED model, and compiled on the
# UNPRUNED model for the speed-up the paper reports as TA. Eval: the folded model over the
# FULL test set, eager vs compiled, pruned and unpruned (IA). Everything is freed
# afterwards so the two workers start with the whole GPU.
import gc, time, torch
from proj.model import build_model, n_params
from proj.lwfednids import (layout, flatten, unflatten_into, client_update,
                            make_optimizer, ACC_KEYS)
from proj.evaluate import fold_bn, load_folded, eval_model
from proj import driver as D

CAL = {}
dev = torch.device("cuda:0"); torch.cuda.set_device(dev)
torch.backends.cudnn.benchmark = True
for _n in ('recompile_limit', 'cache_size_limit'):
    if hasattr(torch._dynamo.config, _n): setattr(torch._dynamo.config, _n, 64)
cache = Path(CFG["cache"])
TX = D._resident(cache / "test_X.f16.npy", dev); TY = D._resident(cache / "test_y.u8.npy", dev)
lo, hi = spans[min(spans, key=lambda c: spans[c][1] - spans[c][0])]   # smallest client
hi = min(hi, lo + 400 * CFG["batch"])                                  # ~400 steps
X = torch.from_numpy(np.load(cache / "train_X.f16.npy", mmap_mode="r")[lo:hi].copy()).to(dev)
Y = torch.from_numpy(np.load(cache / "train_y.u8.npy", mmap_mode="r")[lo:hi].copy()).to(dev)

def one_client(Sc, Se, cfg):
    opt = make_optimizer(Se, cfg["lr"], cfg, fused=True)
    sc = torch.amp.GradScaler("cuda"); sc.scale(torch.zeros(1, device=dev))
    g = torch.Generator(device=dev); g.manual_seed(1)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    acc, n = client_update(Sc, Se, opt, sc, X, Y, 0, hi - lo, cfg, g)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n * 1000, n, int(acc[len(ACC_KEYS)].item())

def measure(tag, cfg):
    """eager and compiled ms/step of one client, then eval rows/s of the folded model."""
    torch.manual_seed(cfg["seed"])
    Se = build_model(cfg).to(dev).train()
    fk, ik, _ = layout(Se); s0, i0 = flatten(Se, fk, ik)
    reset = lambda: unflatten_into(Se, s0, i0, fk, ik)
    ms, n, sk = one_client(Se, Se, cfg)
    CAL[f"{tag}_params"] = n_params(Se)
    CAL[f"{tag}_train_eager_ms_per_step"] = ms
    print(f"{tag:>8} train eager   : {ms:.2f} ms/step ({n} steps, {sk} skipped) at batch {cfg['batch']}, {n_params(Se):,} params")
    reset(); t0 = time.perf_counter()
    Sc = D._compile_train(Se, cfg, dev, X[:cfg["batch"]].float(), Y[:cfg["batch"]].long())
    CAL[f"{tag}_compile_seconds"] = time.perf_counter() - t0
    CAL[f"{tag}_train_backend"] = "compiled" if Sc is not Se else "eager"
    if Sc is not Se:
        reset(); one_client(Sc, Se, cfg)                          # warm the graphs
        reset(); ms, n, sk = one_client(Sc, Se, cfg)
        CAL[f"{tag}_train_compiled_ms_per_step"] = ms
        print(f"{tag:>8} train compiled: {ms:.2f} ms/step ({n} steps, {sk} skipped) | "
              f"{CAL[f'{tag}_train_eager_ms_per_step']/ms:.2f}x | compile+gate {CAL[f'{tag}_compile_seconds']:.0f}s")
    del Sc
    Te = fold_bn(build_model(cfg).to(dev)); load_folded(Te, Se)
    for eb in (16384, 32768):
        c = dict(cfg, eval_batch=eb)
        eval_model(Te, Te, TX[:4 * eb], TY[:4 * eb], c)              # cuDNN autotune, once
        torch.cuda.synchronize(); t0 = time.perf_counter()
        cm, nf, _ = eval_model(Te, Te, TX, TY, c); torch.cuda.synchronize()
        r = TX.shape[0] / (time.perf_counter() - t0); CAL[f"{tag}_eval_eager_folded_{eb}"] = r
        print(f"{tag:>8} eval eager-folded  batch {eb:>5}: {r:,.0f} rows/s")
        if cfg["compile"]:
            Tc = D._compile_eval(Te, c, dev, TX[:eb].float())
            if Tc is not Te:
                eval_model(Tc, Te, TX[:4 * eb], TY[:4 * eb], c)          # warm
                torch.cuda.synchronize(); t0 = time.perf_counter()
                cm2, nf2, _ = eval_model(Tc, Te, TX, TY, c); torch.cuda.synchronize()
                r = TX.shape[0] / (time.perf_counter() - t0); CAL[f"{tag}_eval_compiled_folded_{eb}"] = r
                print(f"{tag:>8} eval compiled-folded batch {eb:>5}: {r:,.0f} rows/s | "
                      f"|dCM|={int((cm2 - cm).abs().sum())} cells of {TX.shape[0]}")
                torch._dynamo.reset()
            del Tc
    del Te, Se, s0
    gc.collect(); torch.cuda.empty_cache(); torch._dynamo.reset()

measure("pruned", CFG)
# the unpruned DAGSNet through the same code path: the paper's baseline for TA / IA
CFG0 = dict(CFG, sparsity=0.0, prune_plan=None, plan_id=None, n_params=395_024)
measure("unpruned", CFG0)
for k in ("train_compiled_ms_per_step", "train_eager_ms_per_step"):
    if f"pruned_{k}" in CAL and f"unpruned_{k}" in CAL:
        CAL[f"TA_{k}"] = CAL[f"unpruned_{k}"] / CAL[f"pruned_{k}"]
for eb in (16384, 32768):
    for k in (f"eval_compiled_folded_{eb}", f"eval_eager_folded_{eb}"):
        if f"pruned_{k}" in CAL and f"unpruned_{k}" in CAL:
            CAL[f"IA_{k}"] = CAL[f"pruned_{k}"] / CAL[f"unpruned_{k}"]
print("TA/IA (unpruned time / pruned time):",
      {k: round(v, 3) for k, v in CAL.items() if k.startswith(("TA_", "IA_"))})
# What a full round would cost from these numbers alone, before any round has run.
_mt = CAL.get("pruned_train_compiled_ms_per_step", CAL["pruned_train_eager_ms_per_step"]) / 1000.0
_re = max(CAL.get("pruned_eval_compiled_folded_16384", 0), CAL.get("pruned_eval_eager_folded_16384", 1))
_steps = sum(-(-(h - l) // CFG["batch"]) for l, h in spans.values()) * CFG["local_epochs"]
CAL["projected_train_sec"] = _steps * _mt / CFG["world_size"]
CAL["projected_eval_sec"] = TX.shape[0] / _re / CFG["world_size"]
CAL["projected_round_sec"] = CAL["projected_train_sec"] + CAL["projected_eval_sec"]
print(f"projected round: train {CAL['projected_train_sec']:.0f}s "
      f"({_steps:,} steps / {CFG['world_size']} GPUs) + eval {CAL['projected_eval_sec']:.0f}s "
      f"(1 model, rows split over {CFG['world_size']} GPUs) = {CAL['projected_round_sec']/60:.1f} min; "
      f"{CFG['rounds']} rounds = {CAL['projected_round_sec']*CFG['rounds']/3600:.1f} h")
del X, Y, TX, TY
gc.collect(); torch.cuda.empty_cache(); torch._dynamo.reset()
CAL["vram_after_free_gb"] = torch.cuda.memory_allocated(dev) / 2**30
print("calibration:", json.dumps({k: (round(v, 3) if isinstance(v, float) else v) for k, v in CAL.items()}))
(C.run_dir(CFG["run_name"]) / "reports" / "calibration.json").write_text(json.dumps(CAL, indent=1))
if run is not None:
    run.summary.update({f"cal_{k}": v for k, v in CAL.items()})


In [ ]:
from proj.driver import run as train, write_manifest
# Raises if this run's checkpoints were trained on different data. The resume gate above
# ran before the decode and could only compare data_id; content_id is the post-decode one.
write_manifest(CFG, CLASS_NAMES, spans,
               y_true_src=Path(CFG["cache"]) / "test_y.u8.npy",
               extra={"fl_root": str(FL_ROOT), "test_root": str(TEST_ROOT),
                      "feature_cols": FEATS, "n_test": CFG["n_test"],
                      "data_id": CFG["data_id"], "content_id": CFG["content_id"],
                      "prune": PRUNE,
                      "scaler": {c: [SCALER[c]["mean"], SCALER[c]["std_used"]]
                                 for c in FEATS}})
# The mask, on its own, next to the numbers it produced.
(C.run_dir(CFG["run_name"]) / "reports" / "prune_plan.json").write_text(
    json.dumps({"plan_id": CFG["plan_id"], "sparsity": CFG["sparsity"],
                "n_params": CFG["n_params"], "plan": CFG["prune_plan"]}, indent=1))
hist = train(CFG, spans, CLASS_NAMES, wandb_run=run, t_origin=T0)
print(f"\ncompleted {len(hist)} rounds this session")


In [ ]:
# Every published number, re-derived from the artifacts on disk. Never from memory.
import csv
from proj.verify import verify_run
from proj.model import build_model
from proj.metrics import METRIC_KEYS

d = C.run_dir(CFG["run_name"])
# y_true from the RUN, not from /kaggle/temp: the cache is gone with the session, and the
# check has to be the same one someone can repeat after downloading the output alone.
ok, lines = verify_run(d, cfg=CFG, build_model=build_model, expect_params=CFG["n_params"],
                       y_true_path=d / "reports" / "y_true.u8.npy", full=True)
print("\n".join(lines))

last = C.last_complete_round(d, C.fingerprint(CFG)) or 0
print(f"\nrounds verified : {last} / {CFG['rounds']}")
if last < CFG["rounds"]:
    print(f"  INCOMPLETE — attach this notebook's output (or a checkpoint dataset of it) to "
          f"the next push and regenerate with --require-resume to continue at round {last + 1}")
rows = [r for r in csv.DictReader(open(d / "history.csv")) if int(r["round"]) <= last]
if rows:
    fin = rows[-1]
    # The headline is the LAST round, fixed before the run. best-f1 is chosen on the test
    # set after seeing it, so it is a description of the curve and not a second result.
    print(f"\nresult at round {fin['round']} (global model, sparsity {CFG['sparsity']}, "
          f"{CFG['n_params']:,} params, {float(fin['model_mib']):.3f} MiB on the wire):")
    for k in METRIC_KEYS: print(f"  {k:<20} {float(fin[k]):.6f}")
    print(f"  loss over clients    mean {float(fin['loss_client_mean']):.4f} "
          f"std {float(fin['loss_client_std']):.4f} "
          f"min {float(fin['loss_client_min']):.4f} max {float(fin['loss_client_max']):.4f}")
    b = max(rows, key=lambda r: float(r["f1_macro"]))
    print(f"\n[descriptive only] best f1_macro {float(b['f1_macro']):.6f} "
          f"at round {b['round']} — picked on test, not a reported result")

# Calibration, from THIS session's rounds only (the CSV would mix in imported rounds).
sec = [float(r["seconds"]) for r in hist]
overhead = (time.monotonic() - T0) - sum(sec)
print(f"\nbackend  : train {CFG.get('backend', '?')} | eval {CFG.get('backend_eval', '?')}")
print(f"session  : {len(hist)} round(s) here | startup+prepack+compile {overhead/60:.1f} min"
      f" | verify and W&B are outside this figure")
if len(sec) < 2:
    print("timing   : need 2 completed rounds to separate startup from steady state; "
          f"got {len(sec)}. No projection.")
else:
    steady = sum(sec[1:]) / len(sec[1:])
    vt = max(float(r.get("vram_train_gb", 0) or 0) for r in hist)
    ve = max(float(r.get("vram_eval_gb", 0) or 0) for r in hist)
    tr = sum(float(r["train_sec"]) for r in hist[1:]) / len(hist[1:])
    ev = sum(float(r["eval_sec"]) for r in hist[1:]) / len(hist[1:])
    print(f"timing   : rounds {[round(x) for x in sec[-3:]]}s | steady {steady:.0f}s/round "
          f"= train {tr:.0f}s + eval {ev:.0f}s + commit")
    print(f"VRAM     : train {vt:.2f} GiB/GPU | eval {ve:.2f} GiB/GPU (of 16)")
    print(f"projected: {CFG['rounds']} rounds = "
          f"{(steady*CFG['rounds'] + overhead)/3600:.2f} h "
          f"({(steady*CFG['rounds'])/3600:.2f} h of rounds + {overhead/3600:.2f} h startup)")
if run is not None: run.finish()
assert ok, "artifact verification FAILED — see the FAIL lines above"
